# AutoTraductor — versión Gradio (autónoma)

**Uso:** activa *GPU* (recomendado: *GPU T4 x2* o *T4*) e *Internet* en el panel derecho → **Restart session** → **Run All** → espera en la última celda a
ver el recuadro con la **URL pública** (`https://xxxx.gradio.live`) → ábrela en el navegador.

No necesita Cloudflare, Vercel, dominio ni ningún token. La URL cambia cada vez que ejecutas la última celda.
Los bloques son de hasta 25 minutos (un video menor queda en un solo bloque); si Kaggle ofrece *GPU T4 x2*, actívala para que el pipeline use las dos GPU visibles en paralelo en la transcripción y en la síntesis de voz.

**Arquitectura del pipeline:**
- **Transcripción:** Whisper (`large-v3`).
- **Traducción multilingüe:** NLLB-200 distilled 600M (local, ~1.2 GB VRAM).
- **Motores de clonación de voz:** **XTTS v2** como opción predeterminada (17 idiomas) y **Qwen3-TTS Base 0.6B** como alternativa (10 idiomas). Qwen 1.7B permanece deshabilitado.

Secrets opcionales (*Add-ons → Secrets*, con acceso activado para este notebook):

| Secret | Uso |
|---|---|
| `GRADIO_PASSWORD` | Si existe, la interfaz pide usuario `admin` y esa clave (recomendado: la URL es pública) |
| `YT_COOKIES` | Cookies de YouTube (formato Netscape) si YouTube bloquea la descarga |
| `KAGGLE_USERNAME` / `KAGGLE_KEY` | Guardar modelos y estado en el Dataset `autotraductor` |

Las celdas que solo escriben archivos no muestran salida: es normal.

## 1. Configuración y secrets (todos opcionales)

In [ ]:
import os, sys, subprocess, shutil, time, pathlib, urllib.request

ENABLE_CLONE_MODE = True       # Instala XTTS v2 y dependencias de voz
ENABLE_QWEN_TTS = True         # Instala Qwen3-TTS Base 0.6B en un overlay aislado
STORAGE_LIMIT_GB = 15          # /kaggle/working tiene ~20 GB
APP_DIR = pathlib.Path("/kaggle/working/autotraductor-app")
DATA_DIR = pathlib.Path("/kaggle/working/autotraductor")
HF_HOME = pathlib.Path("/kaggle/tmp/hf-cache")
COOKIES_FILE = pathlib.Path("/kaggle/tmp/yt_cookies.txt")
for p in (APP_DIR, DATA_DIR, HF_HOME, COOKIES_FILE.parent):
    p.mkdir(parents=True, exist_ok=True)

SECRETS = {n: None for n in ["GRADIO_PASSWORD", "YT_COOKIES", "KAGGLE_USERNAME", "KAGGLE_KEY"]}
try:
    from kaggle_secrets import UserSecretsClient
    _client = UserSecretsClient()
    for _name in SECRETS:
        try:
            _value = _client.get_secret(_name)
            SECRETS[_name] = _value.strip() if _value else None
        except Exception:
            pass
except ImportError:
    print("kaggle_secrets no disponible: se continúa sin secrets (todos son opcionales).")

status = lambda n: "OK" if SECRETS[n] else "no definido"
print("Secrets opcionales: " + ", ".join(f"{n}={status(n)}" for n in SECRETS))

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
print("GPU(s) visibles; se detectan todas automáticamente:", gpu.stdout.strip() if gpu.returncode == 0 else "NO DETECTADA (Settings > Accelerator > GPU)")
try:
    urllib.request.urlopen("https://pypi.org/simple/pip/", timeout=10)
    print("Internet: OK")
except Exception:
    raise RuntimeError("Internet desactivado. Activa Settings > Internet y ejecuta Restart session + Run All.")


## 2. Dependencias base

In [ ]:
def pip(*args, check=True):
    result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-input", *args], capture_output=True, text=True)
    if check and result.returncode != 0:
        print(result.stderr[-2000:])
        raise RuntimeError(f"pip install falló: {' '.join(args)}")
    return result.returncode == 0

print("Instalando dependencias base…")
pip("--upgrade", "yt-dlp[default]")
pip("gradio>=5,<6", "faster-whisper>=1.1", "soundfile", "kaggle", "sentencepiece")
pip("deno", check=False)  # runtime JS que yt-dlp usa para resolver YouTube

import importlib.metadata as md
for pkg in ["torch", "transformers", "yt-dlp", "gradio", "faster-whisper", "sentencepiece"]:
    try:
        print(f"  {pkg}=={md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"  {pkg}: NO INSTALADO")

print("Dependencias base: OK")


## 3. Modo clonación XTTS v2 (opción predeterminada)

In [ ]:
import importlib.metadata as _clone_md

CLONE_READY = False
CLONE_INSTALL_LOG = pathlib.Path("/kaggle/tmp/clone_install.log")
CLONE_INSTALL_LOG.parent.mkdir(parents=True, exist_ok=True)


def _clone_log(message, mode="a"):
    with CLONE_INSTALL_LOG.open(mode, encoding="utf-8") as log:
        log.write(message.rstrip() + "\n")


def _clone_version(package):
    try:
        return _clone_md.version(package)
    except _clone_md.PackageNotFoundError:
        return None


def _clone_read_log():
    try:
        return CLONE_INSTALL_LOG.read_text(encoding="utf-8", errors="replace")
    except OSError as exc:
        return f"No se pudo leer el registro de instalación: {type(exc).__name__}: {exc}"


CLONE_INSTALL_LOG.write_text("", encoding="utf-8")
_clone_log(f"Python: {sys.version}")
_clone_log("Comprobando e instalando coqui-tts para XTTS v2 y dependencias de clonación.")
if ENABLE_CLONE_MODE:
    versions = {name: _clone_version(name) for name in ("torch", "torchaudio", "transformers", "numpy")}
    for name, version in versions.items():
        _clone_log(f"{name}: {version or 'NO INSTALADO'}")
    print("Instalando clonación de voz (Coqui XTTS v2 predeterminado + faster-whisper + NLLB-200)…")
    print("  Versiones actuales:", ", ".join(f"{name}={version or 'no instalado'}" for name, version in versions.items()))

    torch_version = versions["torch"]
    if sys.version_info < (3, 10) or sys.version_info >= (3, 15):
        _clone_log(f"ERROR: coqui-tts 0.27.5 requiere Python >=3.10,<3.15; actual: {sys.version.split()[0]}")
        install_returncode = 1
        pytorch_index = None
    elif not torch_version:
        _clone_log("ERROR: PyTorch no está instalado.")
        install_returncode = 1
        pytorch_index = None
    elif not versions["torchaudio"]:
        _clone_log("ERROR: torchaudio no está instalado.")
        install_returncode = 1
        pytorch_index = None
    else:
        try:
            import torch as _clone_torch

            cuda_version = _clone_torch.version.cuda
            if cuda_version:
                cuda_parts = cuda_version.split(".")
                pytorch_index = f"https://download.pytorch.org/whl/cu{int(cuda_parts[0])}{int(cuda_parts[1])}"
            else:
                pytorch_index = "https://download.pytorch.org/whl/cpu"
        except Exception as exc:
            pytorch_index = None
            _clone_log(f"ERROR: no se pudo inspeccionar el build de PyTorch: {type(exc).__name__}: {exc}")

        base_torch_version = torch_version.split("+", 1)[0]
        version_parts = base_torch_version.split(".")
        torch_major, torch_minor = int(version_parts[0]), int(version_parts[1])
        requirements = [
            "coqui-tts==0.27.5",
            "transformers==4.57.6",
            "faster-whisper>=1.1",
            "sentencepiece",
        ]
        if (torch_major, torch_minor) == (2, 9):
            requirements.append("torchcodec>=0.9,<0.10")
        elif (torch_major, torch_minor) == (2, 10):
            requirements.append("torchcodec>=0.10,<0.11")
        elif (torch_major, torch_minor) >= (2, 11):
            requirements.append("torchcodec>=0.11")

        constraints = pathlib.Path("/kaggle/tmp/clone_constraints.txt")
        pinned = [f"torch=={torch_version}", f"torchaudio=={versions['torchaudio']}"]
        if versions["numpy"]:
            pinned.append(f"numpy=={versions['numpy']}")
        constraints.write_text("\n".join(pinned) + "\n", encoding="utf-8")
        command = [
            sys.executable, "-m", "pip", "install", "--no-input", "--disable-pip-version-check",
            "--progress-bar", "off", "--constraint", str(constraints),
        ]
        if pytorch_index:
            command += ["--extra-index-url", pytorch_index]
        command += requirements
        _clone_log("\n$ " + " ".join(command) + "\n")
        print("  Fijando transformers==4.57.6 compatible con XTTS v2 y NLLB-200; torch no se reinstala.")
        try:
            with CLONE_INSTALL_LOG.open("a", encoding="utf-8") as log:
                install = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT, text=True)
            install_returncode = install.returncode
        except Exception as exc:
            install_returncode = 1
            _clone_log(f"ERROR al ejecutar pip: {type(exc).__name__}: {exc}")

    if install_returncode == 0:
        print("  pip terminó correctamente. Comprobando XTTS v2, Whisper, NLLB-200 y lectura de audio…")
    else:
        print("  La instalación o una precondición falló. Diagnóstico completo (también guardado en /kaggle/tmp/clone_install.log):")
        print(_clone_read_log())

    smoke_test = r"""import importlib.metadata as md
import shutil
import tempfile
from pathlib import Path
import numpy as np
import soundfile as sf
import torch
import torchaudio
import transformers
from faster_whisper import WhisperModel
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from TTS.api import TTS as CoquiTTS
import TTS.tts.models.xtts

xtts_name = "tts_models/multilingual/multi-dataset/xtts_v2"
assert xtts_name in CoquiTTS.list_models(), f"XTTS v2 no aparece en el catálogo de Coqui: {xtts_name}"
print("XTTS v2 catálogo: OK")
if shutil.which("ffmpeg"):
    with tempfile.TemporaryDirectory() as tmp:
        audio_path = Path(tmp) / "probe.wav"
        sf.write(audio_path, np.zeros(1600, dtype=np.float32), 16000, subtype="PCM_16")
        audio, sample_rate = torchaudio.load(str(audio_path))
        assert sample_rate == 16000 and audio.shape[-1] > 0, "torchaudio no pudo leer el audio de referencia"
    print("torchaudio / referencia de voz: OK")
else:
    print("FFmpeg aún no está instalado; la lectura de audio se comprueba al arrancar la app.")
print("Whisper + NLLB-200 + XTTS v2: OK")
for name in ("torch", "torchaudio", "transformers", "numpy", "coqui-tts", "faster-whisper"):
    try:
        print(f"  {name}=={md.version(name)}")
    except md.PackageNotFoundError:
        print(f"  {name}: NO INSTALADO")
"""
    try:
        check = subprocess.run(
            [sys.executable, "-c", smoke_test], capture_output=True, text=True,
            env={**os.environ, "COQUI_TOS_AGREED": "1"},
        )
        _clone_log("\n--- Smoke test imports/runtime ---\n")
        _clone_log(check.stdout or "(sin stdout)")
        if check.stderr:
            _clone_log(check.stderr)
        CLONE_READY = install_returncode == 0 and check.returncode == 0
        if check.returncode != 0:
            print("  Prueba de importación/runtime fallida:")
            print((check.stdout or "") + (check.stderr or ""))
    except Exception as exc:
        CLONE_READY = False
        _clone_log(f"ERROR al ejecutar la prueba: {type(exc).__name__}: {exc}")
        print(f"  No se pudo completar la prueba XTTS/Whisper/NLLB: {type(exc).__name__}: {exc}")

    if not CLONE_READY:
        print("  El detalle completo está en /kaggle/tmp/clone_install.log.")
else:
    _clone_log("Clonación desactivada por ENABLE_CLONE_MODE=False.")
    print("Clonación omitida por ENABLE_CLONE_MODE=False.")

print("Modo clonación XTTS v2:", "OK" if CLONE_READY else "no disponible")


## 4. Qwen3-TTS Base 0.6B (motor alternativo)

In [ ]:
QWEN_TTS_READY = False
QWEN_TTS_ERROR = "Qwen3-TTS no instalado (motor alternativo opcional)."
QWEN_TTS_PACKAGE = pathlib.Path("/kaggle/tmp/autotraductor-qwen-package")
QWEN_TTS_LOG = pathlib.Path("/kaggle/tmp/qwen_tts_install.log")
QWEN_TTS_LOG.parent.mkdir(parents=True, exist_ok=True)

if ENABLE_CLONE_MODE and globals().get("ENABLE_QWEN_TTS", False):
    print("Preparando Qwen3-TTS Base 0.6B en un overlay aislado; no se cambiarán torch ni Transformers compartidos…")
    try:
        shutil.rmtree(QWEN_TTS_PACKAGE, ignore_errors=True)
        QWEN_TTS_PACKAGE.mkdir(parents=True, exist_ok=True)
        qwen_command = [
            sys.executable, "-m", "pip", "install", "--no-input", "--disable-pip-version-check",
            "--progress-bar", "off", "--no-deps", "--target", str(QWEN_TTS_PACKAGE),
            "qwen-tts==0.1.1", "accelerate==1.12.0", "librosa==0.11.0", "soxr==0.5.0.post1",
        ]
        with QWEN_TTS_LOG.open("w", encoding="utf-8") as log:
            install = subprocess.run(qwen_command, stdout=log, stderr=subprocess.STDOUT, text=True, timeout=900)

        if install.returncode != 0:
            QWEN_TTS_ERROR = QWEN_TTS_LOG.read_text(encoding="utf-8", errors="replace")[-1800:]
            print("  No se pudo instalar el overlay de Qwen. XTTS v2 no se modificó.")
            print(QWEN_TTS_ERROR)
        else:
            qwen_core_init = QWEN_TTS_PACKAGE / "qwen_tts" / "core" / "__init__.py"
            core_text = qwen_core_init.read_text(encoding="utf-8")
            core_text = "\n".join(
                line for line in core_text.splitlines()
                if not line.startswith("from .tokenizer_25hz.")
            ) + "\n"
            qwen_core_init.write_text(core_text, encoding="utf-8")

            tokenizer_wrapper = QWEN_TTS_PACKAGE / "qwen_tts" / "inference" / "qwen3_tts_tokenizer.py"
            wrapper_text = tokenizer_wrapper.read_text(encoding="utf-8")
            v1_imports = """from ..core import (
    Qwen3TTSTokenizerV1Config,
    Qwen3TTSTokenizerV1Model,
    Qwen3TTSTokenizerV2Config,
    Qwen3TTSTokenizerV2Model,
)"""
            v2_imports = "from ..core import Qwen3TTSTokenizerV2Config, Qwen3TTSTokenizerV2Model"
            v1_registration = """        AutoConfig.register("qwen3_tts_tokenizer_25hz", Qwen3TTSTokenizerV1Config)
        AutoModel.register(Qwen3TTSTokenizerV1Config, Qwen3TTSTokenizerV1Model)
"""
            if v1_imports not in wrapper_text or v1_registration not in wrapper_text:
                raise RuntimeError("La estructura de qwen-tts cambió; se omite Qwen para proteger las dependencias compartidas.")
            wrapper_text = wrapper_text.replace(v1_imports, v2_imports, 1).replace(
                v1_registration, "        # Los checkpoints Base instalados usan exclusivamente el tokenizer 12 Hz.\n", 1
            )
            tokenizer_wrapper.write_text(wrapper_text, encoding="utf-8")

            tokenizer_file = QWEN_TTS_PACKAGE / "qwen_tts" / "core" / "tokenizer_12hz" / "modeling_qwen3_tts_tokenizer_v2.py"
            tokenizer_text = tokenizer_file.read_text(encoding="utf-8")
            original_import = "from transformers.utils.generic import check_model_inputs"
            compatibility_import = """try:
    from transformers.utils.generic import check_model_inputs
except ImportError:
    def check_model_inputs(function=None, *_args, **_kwargs):
        if callable(function):
            return function
        return lambda actual_function: actual_function
"""
            if original_import in tokenizer_text:
                tokenizer_text = tokenizer_text.replace(original_import, compatibility_import, 1)
                tokenizer_file.write_text(tokenizer_text, encoding="utf-8")

            qwen_smoke_test = r"""from qwen_tts import Qwen3TTSModel
from qwen_tts.core.models import Qwen3TTSForConditionalGeneration
from qwen_tts.core.tokenizer_12hz.modeling_qwen3_tts_tokenizer_v2 import Qwen3TTSTokenizerV2Model
print("Qwen3-TTS Base 0.6B imports: OK")
"""
            qwen_env = {**os.environ, "PYTHONPATH": str(QWEN_TTS_PACKAGE) + os.pathsep + os.environ.get("PYTHONPATH", "")}
            qwen_env["AUTOTRADUCTOR_QWEN_TTS_PACKAGE"] = str(QWEN_TTS_PACKAGE)
            try:
                check = subprocess.run(
                    [sys.executable, "-c", qwen_smoke_test], capture_output=True, text=True,
                    env=qwen_env, timeout=240,
                )
                QWEN_TTS_READY = check.returncode == 0
                if not QWEN_TTS_READY:
                    QWEN_TTS_ERROR = ((check.stderr or "") + "\n" + (check.stdout or "")).strip()[-1800:]
                    with QWEN_TTS_LOG.open("a", encoding="utf-8") as log:
                        log.write("\n--- Qwen import smoke test ---\n" + QWEN_TTS_ERROR + "\n")
                    print("  Qwen no pasó la prueba de importación. XTTS v2 sigue como opción predeterminada.")
                    print(QWEN_TTS_ERROR)
            except Exception as exc:
                QWEN_TTS_ERROR = f"{type(exc).__name__}: {exc}"
                print("  Falló la prueba de importación de Qwen:", QWEN_TTS_ERROR)

        if QWEN_TTS_READY:
            QWEN_TTS_ERROR = ""
            print("  Qwen3-TTS Base 0.6B listo como motor alternativo.")
        else:
            print("  Qwen3-TTS no disponible. XTTS v2 permanece activo.")
    except Exception as exc:
        QWEN_TTS_READY = False
        QWEN_TTS_ERROR = f"{type(exc).__name__}: {exc}"
        print("  Falló la preparación aislada de Qwen:", QWEN_TTS_ERROR)
else:
    reason = "ENABLE_CLONE_MODE=False" if not ENABLE_CLONE_MODE else "ENABLE_QWEN_TTS=False"
    QWEN_TTS_ERROR = f"Qwen3-TTS omitido ({reason})."
    print(QWEN_TTS_ERROR)

print("Qwen3-TTS Base 0.6B:", "OK" if QWEN_TTS_READY else "no disponible")


## 5. ffmpeg

In [ ]:
if shutil.which("ffmpeg") is None:
    print("Instalando ffmpeg…")
    subprocess.run("apt-get -qq update && apt-get -qq install -y ffmpeg", shell=True, check=True)
print("ffmpeg:", subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True).stdout.splitlines()[0])


## 6. Cookies y persistencia (opcional)

In [ ]:
if SECRETS["YT_COOKIES"]:
    COOKIES_FILE.write_text(SECRETS["YT_COOKIES"] + "\n")
    COOKIES_FILE.chmod(0o600)

restored = []
candidates = [p for p in pathlib.Path("/kaggle/input").glob("**/autotraductor") if p.is_dir()][:1] if pathlib.Path("/kaggle/input").exists() else []
if candidates:
    src = candidates[0]
    if (src / "hf").is_dir():
        shutil.copytree(src / "hf", HF_HOME, dirs_exist_ok=True); restored.append("modelos HF")
    if (src / "tts").is_dir():
        shutil.copytree(src / "tts", pathlib.Path.home() / ".local/share/tts", dirs_exist_ok=True); restored.append("XTTS")
    if (src / "jobs").is_dir() and not any((DATA_DIR / "jobs").glob("*")):
        shutil.copytree(src / "jobs", DATA_DIR / "jobs", dirs_exist_ok=True); restored.append("trabajos")
print("Persistencia:", ("restaurado " + ", ".join(restored) + f" desde {src}") if restored else "sin Dataset 'autotraductor' adjunto (opcional)")


## 7. Código de la app
Estas celdas escriben archivos y no muestran salida.

In [ ]:
%%writefile {APP_DIR}/jobstore.py
"""Durable job state stored as JSON files so work resumes after a Kaggle restart."""

from __future__ import annotations

import json
import os
import threading
import time
from contextlib import contextmanager
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Optional

DATA_DIR = Path(os.environ.get("AUTOTRADUCTOR_DATA_DIR", "/kaggle/working/autotraductor"))
JOBS_DIR = DATA_DIR / "jobs"

_locks: dict[str, threading.RLock] = {}
_locks_guard = threading.Lock()

ACTIVE_STATUSES = ("queued", "downloading", "processing", "finalizing")


def ensure_dirs() -> None:
    JOBS_DIR.mkdir(parents=True, exist_ok=True)


@contextmanager
def job_lock(job_id: str):
    with _locks_guard:
        lock = _locks.setdefault(job_id, threading.RLock())
    with lock:
        yield


@dataclass
class Job:
    id: str
    url: str
    source_lang: str
    target_lang: str
    mode: str = "clone"
    tts_engine: str = "xtts"
    status: str = "queued"
    title: Optional[str] = None
    duration: float = 0.0
    chunk_seconds: int = 25 * 60
    chunks: list = field(default_factory=list)
    current_chunk: Optional[int] = None
    current_chunk_progress: float = 0.0
    error: Optional[str] = None
    result_ready: bool = False
    result_downloaded: bool = False
    result_bytes: int = 0
    created_at: float = field(default_factory=time.time)
    updated_at: float = field(default_factory=time.time)
    started_at: Optional[float] = None
    run_started_at: Optional[float] = None
    elapsed_seconds: float = 0.0
    # Optional Gradio processing window; None/None means process the full video.
    test_start_seconds: Optional[float] = None
    test_end_seconds: Optional[float] = None
    # Detailed timings per stage
    timings: dict = field(default_factory=lambda: {
        "download_seconds": 0.0,
        "transcribe_seconds": 0.0,
        "translation_seconds": 0.0,
        "tts_seconds": 0.0,
        "mux_seconds": 0.0,
        "model_download_seconds": 0.0,
    })

    @classmethod
    def new(
        cls, job_id: str, url: str, source_lang: str, target_lang: str, mode: str = "clone", tts_engine: Optional[str] = None
    ) -> "Job":
        if tts_engine is None:
            tts_engine = "xtts"
        return cls(id=job_id, url=url, source_lang=source_lang, target_lang=target_lang, mode=mode, tts_engine=tts_engine)

    @property
    def dir(self) -> Path:
        return JOBS_DIR / self.id

    def processed_seconds(self) -> float:
        done = sum(c["end"] - c["start"] for c in self.chunks if c["status"] == "completed")
        if self.current_chunk is not None and 0 <= self.current_chunk < len(self.chunks):
            chunk = self.chunks[self.current_chunk]
            if chunk["status"] == "processing":
                done += (chunk["end"] - chunk["start"]) * self.current_chunk_progress
        return min(done, self.duration) if self.duration else done

    def elapsed(self, now: Optional[float] = None) -> float:
        """Wall-clock time accumulated while the worker owns the job; queue wait is excluded."""
        total = max(0.0, float(self.elapsed_seconds or 0.0))
        if self.run_started_at is not None:
            total += max(0.0, (time.time() if now is None else now) - self.run_started_at)
        return total

    def public(self) -> dict:
        data = asdict(self)
        # Internal Gradio range controls are intentionally not part of the public API.
        data.pop("test_start_seconds", None)
        data.pop("test_end_seconds", None)
        processed = self.processed_seconds()
        data["processed_seconds"] = round(processed, 1)
        data["elapsed_seconds"] = round(self.elapsed(), 1)
        data["progress"] = round(100 * processed / self.duration, 1) if self.duration else 0.0
        if self.status == "completed":
            data["progress"] = 100.0
        data["chunks_completed"] = sum(1 for c in self.chunks if c["status"] == "completed")
        data["chunks_total"] = len(self.chunks)
        # Calculate queue position if job is waiting in queue
        if self.status == "queued":
            all_jobs = list_jobs()
            queued = [j for j in all_jobs if j.status == "queued"]
            # order by created_at ascending (FIFO)
            queued.sort(key=lambda j: j.created_at)
            try:
                idx = [j.id for j in queued].index(self.id)
                data["queue_position"] = idx + 1
                data["queue_total"] = len(queued)
            except ValueError:
                data["queue_position"] = 1
                data["queue_total"] = 1
        else:
            data["queue_position"] = None
            data["queue_total"] = None
        # Format timings rounded
        raw_timings = data.get("timings") or {}
        data["timings"] = {
            k: round(float(v or 0.0), 2)
            for k, v in raw_timings.items()
        }
        return data


def _job_file(job_id: str) -> Path:
    return JOBS_DIR / job_id / "job.json"


def save_job(job: Job) -> None:
    job.updated_at = time.time()
    path = _job_file(job.id)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(asdict(job), ensure_ascii=False, indent=2))
    os.replace(tmp, path)


def load_job(job_id: str) -> Optional[Job]:
    path = _job_file(job_id)
    if not path.is_file():
        return None
    try:
        raw = json.loads(path.read_text())
        if not isinstance(raw, dict):
            return None
        if "timings" not in raw or not isinstance(raw["timings"], dict):
            raw["timings"] = {
                "download_seconds": 0.0,
                "transcribe_seconds": 0.0,
                "translation_seconds": 0.0,
                "tts_seconds": 0.0,
                "mux_seconds": 0.0,
                "model_download_seconds": 0.0,
            }
        # Allow older jobs without error or with older defaults
        if "tts_engine" not in raw:
            raw["tts_engine"] = "xtts"
        return Job(**raw)
    except (json.JSONDecodeError, TypeError):
        return None


def list_jobs() -> list[Job]:
    if not JOBS_DIR.is_dir():
        return []
    jobs = [load_job(p.name) for p in JOBS_DIR.iterdir() if p.is_dir()]
    return sorted((j for j in jobs if j), key=lambda j: j.created_at, reverse=True)


def start_run(job_id: str, now: Optional[float] = None) -> Optional[Job]:
    """Start (or resume) active-time accounting when the worker actually picks up a job."""
    with job_lock(job_id):
        job = load_job(job_id)
        if job is None:
            return None
        instant = time.time() if now is None else now
        if job.started_at is None:
            job.started_at = instant
        if job.run_started_at is None:
            job.run_started_at = instant
        save_job(job)
        return job


def stop_run(job_id: str, now: Optional[float] = None) -> Optional[Job]:
    """Persist this worker run's elapsed time, e.g. before a retry or Kaggle restart."""
    with job_lock(job_id):
        job = load_job(job_id)
        if job is None or job.run_started_at is None:
            return job
        instant = time.time() if now is None else now
        job.elapsed_seconds = max(0.0, float(job.elapsed_seconds or 0.0)) + max(0.0, instant - job.run_started_at)
        job.run_started_at = None
        save_job(job)
        return job


def update(job_id: str, **changes) -> Optional[Job]:
    with job_lock(job_id):
        job = load_job(job_id)
        if job is None:
            return None
        for key, value in changes.items():
            setattr(job, key, value)
        save_job(job)
        return job


def update_chunk(job_id: str, index: int, **changes) -> Optional[Job]:
    with job_lock(job_id):
        job = load_job(job_id)
        if job is None:
            return None
        job.chunks[index].update(changes)
        save_job(job)
        return job


def add_timing(job_id: str, key: str, duration: float) -> Optional[Job]:
    with job_lock(job_id):
        job = load_job(job_id)
        if job is None:
            return None
        if not hasattr(job, "timings") or not isinstance(job.timings, dict):
            job.timings = {}
        job.timings[key] = round(float(job.timings.get(key, 0.0)) + max(0.0, float(duration)), 2)
        save_job(job)
        return job



In [ ]:
%%writefile {APP_DIR}/worker.py
"""Single background worker: processes one job at a time and resumes interrupted jobs."""

from __future__ import annotations

import os
import queue
import tempfile
import threading
import traceback
from pathlib import Path
from typing import Optional

import jobstore

_queue: "queue.Queue[str]" = queue.Queue()
_queued: set[str] = set()
_thread: Optional[threading.Thread] = None
_current: Optional[str] = None
_cancelled: set[str] = set()
_guard = threading.Lock()


class JobCancelled(Exception):
    pass


_mode_probe_lock = threading.Lock()
_mode_probe_results: dict[tuple[str, str], tuple[bool, Optional[str]]] = {}
_qwen_probe_lock = threading.Lock()
_qwen_probe_result: Optional[tuple[bool, Optional[str]]] = None


def mode_status(mode: str = "clone", tts_engine: str = "xtts") -> tuple[bool, Optional[str]]:
    """Probe the selected voice cloning engine once per process."""
    # "fast" is accepted for backward compatibility and mapped to clone with XTTS v2
    effective_mode = "clone" if mode in {"clone", "fast"} else mode
    if effective_mode != "clone":
        return False, f"Modo desconocido: {mode}"
    if tts_engine not in {"xtts", "qwen06"}:
        return False, f"Motor de voz desconocido: {tts_engine}"
    cache_key = (effective_mode, tts_engine)

    with _mode_probe_lock:
        if cache_key in _mode_probe_results:
            return _mode_probe_results[cache_key]

        try:
            import torch  # noqa: F401
            from transformers import AutoModelForSeq2SeqLM, AutoTokenizer  # noqa: F401
            from faster_whisper import WhisperModel  # noqa: F401
            import soundfile as sf
            import numpy as np

            if tts_engine == "xtts":
                from TTS.api import TTS as CoquiTTS
                import torchaudio
                import TTS.tts.models.xtts  # noqa: F401

                xtts_model = "tts_models/multilingual/multi-dataset/xtts_v2"
                if xtts_model not in CoquiTTS.list_models():
                    raise RuntimeError(f"El catálogo de Coqui no contiene {xtts_model}")
                # XTTS loads its speaker reference with torchaudio; test this without downloading weights.
                with tempfile.TemporaryDirectory() as tmp:
                    sample = Path(tmp) / "clone_mode_probe.wav"
                    sf.write(sample, np.zeros(1600, dtype=np.float32), 16000, subtype="PCM_16")
                    waveform, sample_rate = torchaudio.load(str(sample))
                    if sample_rate != 16000 or waveform.shape[-1] == 0:
                        raise RuntimeError("torchaudio no pudo decodificar el audio de referencia de XTTS")
            else:
                available, reason = qwen_tts_status()
                if not available:
                    raise RuntimeError(reason or "Qwen3-TTS no pasó la comprobación de disponibilidad")

            _mode_probe_results[cache_key] = (True, None)
        except Exception as exc:
            _mode_probe_results[cache_key] = (False, f"{type(exc).__name__}: {exc}")

        return _mode_probe_results[cache_key]


def clone_mode_status(tts_engine: str = "xtts") -> tuple[bool, Optional[str]]:
    return mode_status("clone", tts_engine)


def clone_mode_available(tts_engine: str = "xtts") -> bool:
    return clone_mode_status(tts_engine)[0]


def import_qwen_tts_model_class():
    """Import Qwen3-TTS from its optional package overlay without changing shared dependencies."""
    import sys

    package_dir = os.environ.get("AUTOTRADUCTOR_QWEN_TTS_PACKAGE")
    if package_dir and package_dir not in sys.path:
        sys.path.insert(0, package_dir)
    from qwen_tts import Qwen3TTSModel

    return Qwen3TTSModel


def qwen_tts_status() -> tuple[bool, Optional[str]]:
    """Check optional Qwen3-TTS availability once; XTTS remains independent if it is absent."""
    global _qwen_probe_result
    if os.environ.get("AUTOTRADUCTOR_QWEN_TTS_READY") != "1":
        reason = os.environ.get("AUTOTRADUCTOR_QWEN_TTS_ERROR") or "Qwen3-TTS no está instalado o no pasó su prueba de importación."
        return False, reason

    with _qwen_probe_lock:
        if _qwen_probe_result is None:
            try:
                import_qwen_tts_model_class()
                import torch
                if not torch.cuda.is_available():
                    raise RuntimeError("Qwen3-TTS requiere una GPU CUDA en esta interfaz.")
                _qwen_probe_result = (True, None)
            except Exception as exc:
                _qwen_probe_result = (False, f"{type(exc).__name__}: {exc}")
        return _qwen_probe_result


def persistence_available() -> bool:
    import persistence

    return persistence.available()


def is_alive() -> bool:
    return _thread is not None and _thread.is_alive()


def current_job_id() -> Optional[str]:
    return _current


def cancel(job_id: str) -> None:
    _cancelled.add(job_id)


def check_cancelled(job_id: str) -> None:
    if job_id in _cancelled:
        raise JobCancelled(job_id)


def enqueue(job_id: str) -> None:
    with _guard:
        if job_id in _queued:
            return
        _queued.add(job_id)
    _queue.put(job_id)


def _resume_interrupted() -> None:
    # Sort active jobs by created_at ascending so they resume in FIFO order
    active_jobs = [j for j in jobstore.list_jobs() if j.status in jobstore.ACTIVE_STATUSES]
    active_jobs.sort(key=lambda j: j.created_at)

    for listed_job in active_jobs:
        # Close the persisted run interval before requeueing an interrupted job.
        jobstore.stop_run(listed_job.id)
        with jobstore.job_lock(listed_job.id):
            job = jobstore.load_job(listed_job.id)
            if job is None or job.status not in jobstore.ACTIVE_STATUSES:
                continue
            for chunk in job.chunks:
                if chunk["status"] == "processing":
                    chunk["status"] = "pending"
            job.status = "queued"
            job.current_chunk = None
            job.current_chunk_progress = 0.0
            jobstore.save_job(job)
        enqueue(listed_job.id)


def _loop() -> None:
    global _current

    while True:
        job_id = _queue.get()
        with _guard:
            _queued.discard(job_id)

        # Skip if deleted or cancelled while waiting in queue
        if job_id in _cancelled:
            _cancelled.discard(job_id)
            continue
        job = jobstore.load_job(job_id)
        if job is None:
            continue

        _current = job_id
        try:
            jobstore.start_run(job_id)
            import pipeline

            pipeline.run_job(job_id)
        except JobCancelled:
            pass
        except Exception as exc:
            traceback.print_exc()
            jobstore.update(job_id, status="failed", error=f"{type(exc).__name__}: {exc}"[:600])
        finally:
            jobstore.stop_run(job_id)
            _cancelled.discard(job_id)
            _current = None


def start() -> None:
    global _thread
    if is_alive():
        return
    _resume_interrupted()
    _thread = threading.Thread(target=_loop, name="autotraductor-worker", daemon=True)
    _thread.start()


In [ ]:
%%writefile {APP_DIR}/pipeline.py
"""Download -> chunk -> transcribe with Whisper -> translate with NLLB-200 -> synthesize with XTTS/Qwen -> replace audio -> mux. Resumable per chunk."""

from __future__ import annotations

import inspect
import json
import math
import os
import re
import shutil
import subprocess
import tempfile
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Callable, Optional

import numpy as np

import jobstore
import worker

SAMPLE_RATE = 16000
CHUNK_SECONDS = int(os.environ.get("AUTOTRADUCTOR_CHUNK_SECONDS", str(25 * 60)))
MAX_DURATION_HOURS = float(os.environ.get("AUTOTRADUCTOR_MAX_DURATION_HOURS", "8"))
MAX_SEGMENT_SECONDS = 15.0
MAX_CLONE_SPEEDUP = 1.25  # Evita acelerar la voz agresivamente; el exceso se reubica en silencios.
CLONE_REFERENCE_MAX_SECONDS = 12.0
CLONE_REFERENCE_MIN_SINGLE_SECONDS = 6.0
CLONE_REFERENCE_JOIN_GAP_SECONDS = 0.6
CLONE_MIN_GAP_SECONDS = 0.08
NLLB_MODEL = os.environ.get("AUTOTRADUCTOR_NLLB_MODEL", "facebook/nllb-200-distilled-600M")
WHISPER_MODEL = os.environ.get("AUTOTRADUCTOR_WHISPER_MODEL", "large-v3")
XTTS_MODEL = "tts_models/multilingual/multi-dataset/xtts_v2"
COOKIES_FILE = os.environ.get("AUTOTRADUCTOR_YT_COOKIES_FILE", "")

# Exact 17 output language codes advertised for XTTS v2
XTTS_V2_SUPPORTED_LANGUAGES = frozenset({
    "en", "es", "fr", "de", "it", "pt", "pl", "tr", "ru",
    "nl", "cs", "ar", "zh-cn", "ja", "hu", "ko", "hi",
})

# Mapping from our 3-letter codes to XTTS v2 language codes (17)
CODE_TO_XTTS = {
    "eng": "en", "spa": "es", "fra": "fr", "deu": "de", "ita": "it", "por": "pt", "pol": "pl",
    "tur": "tr", "rus": "ru", "nld": "nl", "ces": "cs", "arb": "ar", "cmn": "zh-cn", "jpn": "ja",
    "hun": "hu", "kor": "ko", "hin": "hi",
}

# Mapping from our 3-letter codes to Qwen3-TTS language names (10)
QWEN_TTS_LANGUAGES = {
    "eng": "English", "spa": "Spanish", "deu": "German", "fra": "French", "ita": "Italian",
    "por": "Portuguese", "rus": "Russian", "jpn": "Japanese", "kor": "Korean", "cmn": "Chinese",
}
QWEN_TTS_MODEL_IDS = {
    "qwen06": "Qwen/Qwen3-TTS-12Hz-0.6B-Base",
}
QWEN_SAMPLE_RATE = 24000

# Mapping from Whisper 2-letter detected language codes to our 3-letter codes
WHISPER_TO_CODE = {
    "en": "eng", "es": "spa", "fr": "fra", "de": "deu", "it": "ita", "pt": "por", "nl": "nld",
    "pl": "pol", "tr": "tur", "ru": "rus", "uk": "ukr", "cs": "ces", "ar": "arb", "zh": "cmn",
    "ja": "jpn", "ko": "kor", "hi": "hin", "hu": "hun", "ro": "ron", "sv": "swe", "da": "dan",
    "fi": "fin", "el": "ell", "he": "heb", "id": "ind", "vi": "vie", "th": "tha", "ca": "cat",
    "fa": "pes", "bn": "ben", "ur": "urd", "ta": "tam", "te": "tel", "tl": "tgl", "sk": "slk",
    "bg": "bul", "hr": "hrv", "sr": "srp", "no": "nob", "sw": "swh",
}
CODE_TO_WHISPER = {v: k for k, v in WHISPER_TO_CODE.items()}

# Mapping from 3-letter codes to NLLB-200 language codes (BCP-47 scripts)
CODE_TO_NLLB = {
    "eng": "eng_Latn", "spa": "spa_Latn", "fra": "fra_Latn", "deu": "deu_Latn",
    "ita": "ita_Latn", "por": "por_Latn", "pol": "pol_Latn", "tur": "tur_Latn",
    "rus": "rus_Cyrl", "nld": "nld_Latn", "ces": "ces_Latn", "arb": "arb_Arab",
    "cmn": "zho_Hans", "jpn": "jpn_Jpan", "hun": "hun_Latn", "kor": "kor_Hang",
    "hin": "hin_Deva", "ukr": "ukr_Cyrl", "ron": "ron_Latn", "swe": "swe_Latn",
    "dan": "dan_Latn", "fin": "fin_Latn", "cat": "cat_Latn", "ell": "ell_Grek",
    "heb": "heb_Hebr", "pes": "pes_Arab", "urd": "urd_Arab", "ben": "ben_Beng",
    "vie": "vie_Latn", "tha": "tha_Thai", "ind": "ind_Latn", "tgl": "tgl_Latn",
    "swh": "swh_Latn", "tam": "tam_Taml", "tel": "tel_Telu", "slk": "slk_Latn",
    "bul": "bul_Cyrl", "hrv": "hrv_Latn", "srp": "srp_Cyrl", "nob": "nob_Latn",
}

# Aliases for backward compatibility with older modules
WHISPER_TO_SEAMLESS = WHISPER_TO_CODE
SEAMLESS_TO_WHISPER = CODE_TO_WHISPER
SEAMLESS_TO_XTTS = CODE_TO_XTTS

if frozenset(CODE_TO_XTTS.values()) != XTTS_V2_SUPPORTED_LANGUAGES:
    raise RuntimeError("CODE_TO_XTTS debe mapear exactamente los 17 idiomas que soporta XTTS v2")

_model_lock = threading.Lock()
_whisper = None
_xtts: dict[str, object] = {}
_qwen_tts_models: dict[tuple[str, str], object] = {}
_nllb_translator: Optional["NLLBTranslator"] = None
_nllb_lock = threading.Lock()
_device_locks_guard = threading.Lock()
_device_locks: dict[str, threading.Lock] = {}


def _device() -> str:
    import torch

    return "cuda" if torch.cuda.is_available() else "cpu"


def _torch_device(device_index: int = 0) -> str:
    import torch

    return f"cuda:{device_index}" if torch.cuda.is_available() else "cpu"


def inference_devices() -> list[str]:
    """Return every CUDA device visible to this Kaggle session (e.g. 2x T4), or one CPU fallback."""
    import torch

    if not torch.cuda.is_available():
        return ["cpu"]
    return [f"cuda:{index}" for index in range(max(1, torch.cuda.device_count()))]


def _device_inference_lock(device: str) -> threading.Lock:
    with _device_locks_guard:
        return _device_locks.setdefault(device, threading.Lock())


def _parallel_apply(items, job_id: str, fn, progress: Callable[[float], None], start: float = 0.0, span: float = 1.0):
    """Run independent speech segments across all visible GPUs and return them in source order."""
    items = list(items)
    if not items:
        return []

    devices = inference_devices()
    workers_count = min(len(devices), len(items))
    results = [None] * len(items)

    def run_serial(device: str) -> list:
        for index, item in enumerate(items):
            worker.check_cancelled(job_id)
            results[index] = fn(index, item, device)
            progress(start + span * (index + 1) / len(items))
        return results

    if workers_count <= 1:
        return run_serial(devices[0])

    futures = {}
    parallel_error = None
    with ThreadPoolExecutor(max_workers=workers_count, thread_name_prefix="gpu-segment") as pool:
        try:
            for index, item in enumerate(items):
                worker.check_cancelled(job_id)
                device = devices[index % len(devices)]
                futures[pool.submit(fn, index, item, device)] = index
            completed = 0
            for future in as_completed(futures):
                worker.check_cancelled(job_id)
                results[futures[future]] = future.result()
                completed += 1
                progress(start + span * completed / len(items))
        except Exception as exc:
            parallel_error = exc
            for future in futures:
                future.cancel()

    if parallel_error is not None:
        message = str(parallel_error).lower()
        gpu_failure = any(token in message for token in ("cuda", "device", "cublas", "cudnn", "out of memory"))
        if not gpu_failure:
            raise parallel_error
        print(
            f"[autotraductor] Falló la inferencia multi-GPU ({type(parallel_error).__name__}: {parallel_error}); "
            f"reintentando secuencialmente en {devices[0]}.",
            flush=True,
        )
        results = [None] * len(items)
        return run_serial(devices[0])
    return results


def run(cmd: list[str], timeout: Optional[int] = None) -> subprocess.CompletedProcess:
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    if result.returncode != 0:
        tail = (result.stderr or result.stdout or "").strip().splitlines()[-6:]
        raise RuntimeError(f"{Path(cmd[0]).name} falló: " + " | ".join(tail))
    return result


def probe_duration(path: Path) -> float:
    out = run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "json", str(path)])
    return float(json.loads(out.stdout)["format"]["duration"])


def read_wav(path: Path) -> np.ndarray:
    import soundfile as sf

    audio, _ = sf.read(str(path), dtype="float32")
    return audio if audio.ndim == 1 else audio.mean(axis=1)


def write_wav(path: Path, audio: np.ndarray, sr: int) -> None:
    import soundfile as sf

    sf.write(str(path), audio.astype(np.float32), sr, subtype="PCM_16")


class NLLBTranslator:
    """Independent local multilingual text translator (Meta NLLB-200 distilled 600M)."""

    def __init__(self, device: Optional[str] = None) -> None:
        import torch
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

        self.torch = torch
        self.device = device or (_torch_device() if torch.cuda.is_available() else "cpu")
        self.dtype = torch.float16 if self.device.startswith("cuda") else torch.float32
        print(f"[autotraductor] Inicializando NLLB-200 ({NLLB_MODEL}) en {self.device} ({self.dtype})…", flush=True)
        t0 = time.time()
        self.tokenizer = AutoTokenizer.from_pretrained(NLLB_MODEL)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(
            NLLB_MODEL,
            torch_dtype=self.dtype,
        ).to(self.device).eval()
        print(f"[autotraductor] Traductor NLLB-200 cargado en memoria ({time.time() - t0:.2f}s)", flush=True)

    def translate_batch(self, texts: list[str], src_code: str, tgt_code: str) -> list[str]:
        """Translate a batch of sentences at once with GPU acceleration and inference_mode."""
        if not texts:
            return []
        src_nllb = CODE_TO_NLLB.get(src_code, "eng_Latn")
        tgt_nllb = CODE_TO_NLLB.get(tgt_code, "spa_Latn")
        if src_nllb == tgt_nllb:
            return list(texts)

        non_empty = [(i, t.strip()) for i, t in enumerate(texts) if t.strip()]
        if not non_empty:
            return [""] * len(texts)

        self.tokenizer.src_lang = src_nllb
        forced_bos_token_id = self.tokenizer.convert_tokens_to_ids(tgt_nllb)

        batch_size = 32
        translated_results = [""] * len(texts)
        for b_start in range(0, len(non_empty), batch_size):
            batch_slice = non_empty[b_start : b_start + batch_size]
            b_texts = [item[1] for item in batch_slice]
            inputs = self.tokenizer(
                b_texts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=512,
            )
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
            with self.torch.inference_mode():
                generated_tokens = self.model.generate(
                    **inputs,
                    forced_bos_token_id=forced_bos_token_id,
                    max_new_tokens=512,
                )
            decoded = self.tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
            for j, translated_text in enumerate(decoded):
                orig_idx = batch_slice[j][0]
                translated_results[orig_idx] = translated_text.strip()

        return translated_results


def nllb(device: Optional[str] = None) -> NLLBTranslator:
    global _nllb_translator
    with _nllb_lock:
        if _nllb_translator is None:
            devices = inference_devices()
            target_device = device or devices[0]
            _nllb_translator = NLLBTranslator(target_device)
        return _nllb_translator


def _release_qwen_models_on_device(device: str) -> None:
    """Evict a Qwen checkpoint before a different PyTorch model needs the same GPU."""
    with _model_lock:
        keys = [key for key in _qwen_tts_models if key[0] == device]
        for key in keys:
            del _qwen_tts_models[key]
    if not keys:
        return
    import gc
    import torch

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def whisper():
    global _whisper
    with _model_lock:
        if _whisper is None:
            from faster_whisper import WhisperModel

            devices = inference_devices()
            cuda_indices = [int(device.split(":", 1)[1]) for device in devices if device.startswith("cuda:")]
            device = "cuda" if cuda_indices else "cpu"
            compute_type = "float16" if cuda_indices else "int8"
            if len(cuda_indices) > 1:
                try:
                    _whisper = WhisperModel(
                        WHISPER_MODEL,
                        device=device,
                        device_index=cuda_indices,
                        num_workers=len(cuda_indices),
                        compute_type=compute_type,
                    )
                    print(f"[autotraductor] Whisper distribuido entre {len(cuda_indices)} GPU ({cuda_indices})", flush=True)
                except Exception as exc:
                    print(f"[autotraductor] Whisper multi-GPU no disponible ({type(exc).__name__}: {exc}); usando GPU 0.", flush=True)
            if _whisper is None:
                _whisper = WhisperModel(
                    WHISPER_MODEL,
                    device=device,
                    device_index=cuda_indices[0] if cuda_indices else 0,
                    num_workers=1,
                    compute_type=compute_type,
                )
        return _whisper


def xtts(device: Optional[str] = None):
    target_device = device or _torch_device()
    if target_device == "cuda":
        target_device = "cuda:0"
    _release_qwen_models_on_device(target_device)
    with _model_lock:
        if target_device not in _xtts:
            os.environ.setdefault("COQUI_TOS_AGREED", "1")
            from TTS.api import TTS

            _xtts[target_device] = TTS(XTTS_MODEL).to(target_device)
        return _xtts[target_device]


def qwen_tts(engine: str, device: str):
    model_id = QWEN_TTS_MODEL_IDS.get(engine)
    if model_id is None:
        raise ValueError(f"Motor Qwen3-TTS desconocido: {engine}")
    target_device = device if device.startswith("cuda:") else "cuda:0"
    key = (target_device, model_id)
    _release_other_models_on_device(target_device)
    with _model_lock:
        if key not in _qwen_tts_models:
            import gc
            import torch

            # Keep the selected Qwen checkpoint on this device and release stale model allocations.
            for old_key in [k for k in _qwen_tts_models if k[0] == target_device and k != key]:
                del _qwen_tts_models[old_key]
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            Qwen3TTSModel = worker.import_qwen_tts_model_class()
            dtype = torch.float16
            if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
                dtype = torch.bfloat16
            _qwen_tts_models[key] = Qwen3TTSModel.from_pretrained(
                model_id,
                device_map=target_device,
                dtype=dtype,
                attn_implementation="sdpa",
            )
        return _qwen_tts_models[key]


def _release_other_models_on_device(device: str) -> None:
    """Free cached XTTS speech model before loading a large Qwen checkpoint on that GPU."""
    with _model_lock:
        removed = _xtts.pop(device, None) is not None
    if not removed:
        return
    import gc
    import torch

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def time_stretch(clip: np.ndarray, sr: int, speed: float) -> np.ndarray:
    with tempfile.TemporaryDirectory() as tmp:
        src, dst = Path(tmp) / "in.wav", Path(tmp) / "out.wav"
        write_wav(src, clip, sr)
        run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(src), "-filter:a", f"atempo={speed:.4f}", str(dst)])
        return read_wav(dst)


def select_clone_reference(audio: np.ndarray, segments: list[tuple[float, float, str]]) -> np.ndarray:
    """Choose a coherent, chronological XTTS reference instead of stitching distant utterances."""
    if len(audio) == 0 or not segments:
        return np.empty(0, dtype=np.float32)

    total_samples = len(audio)
    valid_ranges = []
    for start, end, _text in segments:
        first = max(0, min(int(round(start * SAMPLE_RATE)), total_samples))
        last = max(first, min(int(round(end * SAMPLE_RATE)), total_samples))
        if last > first:
            valid_ranges.append((first, last))
    if not valid_ranges:
        return np.empty(0, dtype=np.float32)
    valid_ranges.sort()

    max_samples = int(CLONE_REFERENCE_MAX_SECONDS * SAMPLE_RATE)
    min_single_samples = int(CLONE_REFERENCE_MIN_SINGLE_SECONDS * SAMPLE_RATE)
    first, last = max(valid_ranges, key=lambda span: span[1] - span[0])
    if last - first < min_single_samples:
        best_span = (first, last)
        best_speech_samples = last - first
        max_join_gap = int(CLONE_REFERENCE_JOIN_GAP_SECONDS * SAMPLE_RATE)
        for index, (cluster_start, cluster_end) in enumerate(valid_ranges):
            speech_samples = cluster_end - cluster_start
            for next_start, next_end in valid_ranges[index + 1 :]:
                if next_start - cluster_end > max_join_gap or next_end - cluster_start > max_samples:
                    break
                speech_samples += max(0, next_end - next_start)
                cluster_end = max(cluster_end, next_end)
                if speech_samples > best_speech_samples:
                    best_span = (cluster_start, cluster_end)
                    best_speech_samples = speech_samples
        first, last = best_span

    if last - first > max_samples:
        last = first + max_samples
    return np.asarray(audio[first:last], dtype=np.float32).copy()


def select_qwen_reference(audio: np.ndarray, segments: list[tuple[float, float, str]]) -> tuple[np.ndarray, str]:
    """Use one complete spoken phrase plus its Whisper text as Qwen's voice-clone prompt."""
    total_samples = len(audio)
    candidates = []
    for start, end, text in segments:
        first = max(0, min(int(round(start * SAMPLE_RATE)), total_samples))
        last = max(first, min(int(round(end * SAMPLE_RATE)), total_samples))
        duration = (last - first) / SAMPLE_RATE
        if text.strip() and last > first:
            candidates.append((duration, first, last, text.strip()))
    if not candidates:
        return np.empty(0, dtype=np.float32), ""

    suitable = [item for item in candidates if 3.0 <= item[0] <= CLONE_REFERENCE_MAX_SECONDS]
    if not suitable:
        suitable = [item for item in candidates if item[0] <= CLONE_REFERENCE_MAX_SECONDS] or candidates
    _, first, last, text = max(suitable, key=lambda item: item[0])
    return np.asarray(audio[first:last], dtype=np.float32).copy(), text


def place_clone_segments(
    segments: list[tuple[float, float, str]],
    generated: list[np.ndarray],
    duration: float,
    sr: int,
) -> tuple[np.ndarray, dict[str, float | int]]:
    """Place cloned clips in order, using silence before speeding or cutting speech."""
    output_samples = max(0, int(round(duration * sr)))
    buffer = np.zeros(output_samples, dtype=np.float32)
    cursor = 0.0
    placed_any = False
    stats: dict[str, float | int] = {
        "speedups": 0,
        "max_speedup": 1.0,
        "shifted_segments": 0,
        "max_shift_seconds": 0.0,
        "clipped_at_chunk_end_seconds": 0.0,
    }

    for index, ((source_start, _source_end, _text), wav) in enumerate(zip(segments, generated)):
        clip = np.asarray(wav, dtype=np.float32).reshape(-1)
        if not len(clip):
            continue

        next_start = segments[index + 1][0] if index + 1 < len(segments) else duration
        available = max(float(next_start) - float(source_start), 0.2)
        raw_duration = len(clip) / sr
        if raw_duration > available:
            requested_speed = raw_duration / available
            speed = min(requested_speed, MAX_CLONE_SPEEDUP)
            if speed > 1.01:
                clip = time_stretch(clip, sr, speed)
                actual_speed = raw_duration / max(len(clip) / sr, 1e-6)
                stats["speedups"] += 1
                stats["max_speedup"] = max(float(stats["max_speedup"]), actual_speed)

        earliest_start = cursor + CLONE_MIN_GAP_SECONDS if placed_any else 0.0
        placed_start = max(float(source_start), earliest_start)
        shift = max(0.0, placed_start - float(source_start))
        if shift > 0.02:
            stats["shifted_segments"] += 1
            stats["max_shift_seconds"] = max(float(stats["max_shift_seconds"]), shift)

        begin = max(0, int(round(placed_start * sr)))
        if begin >= output_samples:
            stats["clipped_at_chunk_end_seconds"] += len(clip) / sr
            continue
        writable = min(len(clip), output_samples - begin)
        if writable < len(clip):
            stats["clipped_at_chunk_end_seconds"] += (len(clip) - writable) / sr
            clip = clip[:writable]
        if not len(clip):
            continue

        fade = min(len(clip), int(0.008 * sr))
        if fade:
            clip = clip.copy()
            clip[-fade:] *= np.linspace(1.0, 0.0, fade, dtype=np.float32)
        buffer[begin : begin + len(clip)] += clip
        cursor = placed_start + len(clip) / sr
        placed_any = True

    return normalize(buffer), stats


def normalize(buffer: np.ndarray) -> np.ndarray:
    peak = float(np.max(np.abs(buffer))) if len(buffer) else 0.0
    return buffer * (0.9 / peak) if peak > 0 else buffer


def split_text_for_tts(text: str, max_chars: int = 180) -> list[str]:
    """Split text into manageable chunks so XTTS v2 never exceeds its 400 token limit."""
    text = (text or "").strip()
    if not text:
        return []
    if len(text) <= max_chars:
        return [text]

    sentences = re.split(r'(?<=[.!?\n])\s+', text)
    chunks: list[str] = []

    for s in sentences:
        s = s.strip()
        if not s:
            continue
        if len(s) <= max_chars:
            chunks.append(s)
            continue

        parts = re.split(r'(?<=[,;:—])\s+', s)
        current = ""
        for p in parts:
            p = p.strip()
            if not p:
                continue
            if len(p) > max_chars:
                words = p.split()
                w_current = ""
                for w in words:
                    if len(w_current) + len(w) + 1 <= max_chars:
                        w_current = f"{w_current} {w}".strip()
                    else:
                        if w_current:
                            chunks.append(w_current)
                        w_current = w[:max_chars] if len(w) > max_chars else w
                if w_current:
                    chunks.append(w_current)
            elif len(current) + len(p) + 1 <= max_chars:
                current = f"{current} {p}".strip()
            else:
                if current:
                    chunks.append(current)
                current = p
        if current:
            chunks.append(current)

    return chunks if chunks else [text[:max_chars]]


ProgressFn = Callable[[float], None]


def translate_clone(audio: np.ndarray, job: jobstore.Job, progress: ProgressFn, workdir: Path) -> tuple[np.ndarray, int]:
    engine = getattr(job, "tts_engine", "xtts") or "xtts"
    if engine != "xtts" and engine not in QWEN_TTS_MODEL_IDS:
        raise ValueError(f"Motor de voz no admitido: {engine}")
    is_qwen = engine in QWEN_TTS_MODEL_IDS
    if is_qwen:
        qwen_lang = QWEN_TTS_LANGUAGES.get(job.target_lang)
        if qwen_lang is None:
            raise ValueError(f"Qwen3-TTS no admite el idioma destino '{job.target_lang}'.")
        qwen_ready, qwen_reason = worker.qwen_tts_status()
        if not qwen_ready:
            raise RuntimeError(f"Qwen3-TTS no está disponible: {qwen_reason or 'revisa la instalación del notebook'}")
        xtts_lang = None
        out_sr = QWEN_SAMPLE_RATE
    else:
        xtts_lang = CODE_TO_XTTS.get(job.target_lang)
        if xtts_lang is None:
            raise ValueError(f"XTTS v2 no admite el idioma destino '{job.target_lang}'.")
        out_sr = QWEN_SAMPLE_RATE

    devices = inference_devices()
    asr = whisper()
    if not is_qwen:
        base_tts = xtts(devices[0])
        out_sr = int(getattr(getattr(base_tts, "synthesizer", None), "output_sample_rate", 24000) or 24000)
    duration = len(audio) / SAMPLE_RATE

    # Fase 1: Transcripción con Whisper (cronometrada)
    t_start_whisper = time.time()
    language = CODE_TO_WHISPER.get(job.source_lang) if job.source_lang != "auto" else None
    raw_segments, info = asr.transcribe(audio, language=language, vad_filter=True, beam_size=5)
    segments = []
    for seg in raw_segments:
        worker.check_cancelled(job.id)
        if seg.text.strip():
            segments.append((seg.start, seg.end, seg.text.strip()))
        progress(0.30 * min(seg.end / duration, 1.0))
    whisper_time = time.time() - t_start_whisper
    jobstore.add_timing(job.id, "transcribe_seconds", whisper_time)
    print(f"[autotraductor][whisper] {len(segments)} segmentos transcritos en {whisper_time:.2f}s (idioma detectado: {info.language})", flush=True)

    src_code = WHISPER_TO_CODE.get(info.language, "eng")

    # Selección de audio de referencia para clonación
    if is_qwen:
        reference, reference_text = select_qwen_reference(audio, segments)
    else:
        reference = select_clone_reference(audio, segments)
        reference_text = ""
    if not len(reference):
        return np.zeros(int(duration * out_sr), dtype=np.float32), out_sr
    ref_path = workdir / "speaker_ref.wav"
    write_wav(ref_path, reference, SAMPLE_RATE)

    # Fase 2: Traducción de texto multilingüe con NLLB-200 en lote (cronometrada)
    t_start_trans = time.time()
    source_texts = [seg[2] for seg in segments]
    translator = nllb(devices[0])
    translations = translator.translate_batch(source_texts, src_code, job.target_lang)
    trans_time = time.time() - t_start_trans
    jobstore.add_timing(job.id, "translation_seconds", trans_time)
    print(f"[autotraductor][nllb] {len(translations)} frases traducidas ({src_code} -> {job.target_lang}) en {trans_time:.2f}s", flush=True)

    # Fase 3: Síntesis de voz y clonación (cronometrada, paralela en GPU T4 x2)
    t_start_tts = time.time()
    if is_qwen:
        def synthesize_qwen_segment(index: int, segment: tuple[float, float, str], device: str) -> np.ndarray:
            translated_text = translations[index] if index < len(translations) else ""
            if not translated_text.strip():
                return np.empty(0, dtype=np.float32)
            chunks = split_text_for_tts(translated_text, max_chars=250)
            with _device_inference_lock(device):
                model = qwen_tts(engine, device)
                prompt = model.create_voice_clone_prompt(
                    ref_audio=str(ref_path),
                    ref_text=reference_text,
                    x_vector_only_mode=False,
                )
                audio_pieces = []
                for ch in chunks:
                    if not ch.strip():
                        continue
                    wavs, sample_rate = model.generate_voice_clone(
                        text=ch.strip(),
                        language=qwen_lang,
                        voice_clone_prompt=prompt,
                    )
                    wav = wavs[0]
                    if hasattr(wav, "detach"):
                        wav = wav.detach().float().cpu().numpy()
                    arr = np.asarray(wav, dtype=np.float32).reshape(-1)
                    if len(arr):
                        audio_pieces.append(arr)
                if not audio_pieces:
                    return np.empty(0, dtype=np.float32)
                return np.concatenate(audio_pieces) if len(audio_pieces) > 1 else audio_pieces[0]

        generated = _parallel_apply(segments, job.id, synthesize_qwen_segment, progress, start=0.35, span=0.60)
        timing_engine = "Qwen3-TTS"
    else:
        def synthesize_xtts_segment(index: int, segment: tuple[float, float, str], device: str) -> np.ndarray:
            translated_text = translations[index] if index < len(translations) else ""
            if not translated_text.strip():
                return np.empty(0, dtype=np.float32)
            chunks = split_text_for_tts(translated_text, max_chars=180)
            with _device_inference_lock(device):
                tts = xtts(device)
                audio_pieces = []
                for ch in chunks:
                    if not ch.strip():
                        continue
                    wav = tts.tts(text=ch.strip(), speaker_wav=str(ref_path), language=xtts_lang, split_sentences=True)
                    if wav is not None and len(wav):
                        audio_pieces.append(np.asarray(wav, dtype=np.float32))
                if not audio_pieces:
                    return np.empty(0, dtype=np.float32)
                return np.concatenate(audio_pieces) if len(audio_pieces) > 1 else audio_pieces[0]

        generated = _parallel_apply(segments, job.id, synthesize_xtts_segment, progress, start=0.35, span=0.60)
        timing_engine = "XTTS v2"

    tts_time = time.time() - t_start_tts
    jobstore.add_timing(job.id, "tts_seconds", tts_time)
    print(f"[autotraductor][tts] Generación de voz con {timing_engine} completada en {tts_time:.2f}s", flush=True)

    # Fase 4: Posicionamiento y alineación temporal
    t_start_align = time.time()
    worker.check_cancelled(job.id)
    translated, timing = place_clone_segments(segments, generated, duration, out_sr)
    align_time = time.time() - t_start_align
    jobstore.add_timing(job.id, "mux_seconds", align_time)

    if any(float(timing[key]) > 0 for key in ("speedups", "shifted_segments", "clipped_at_chunk_end_seconds")):
        print(
            f"[autotraductor][clone] Ajuste temporal {timing_engine}: "
            f"acelerados={timing['speedups']} (máx. {float(timing['max_speedup']):.2f}x), "
            f"reubicados={timing['shifted_segments']} (desfase máx. {float(timing['max_shift_seconds']):.2f}s), "
            f"recorte al final del bloque={float(timing['clipped_at_chunk_end_seconds']):.2f}s.",
            flush=True,
        )
    return translated, out_sr


def download_source(job: jobstore.Job) -> Path:
    workdir = job.dir
    existing = workdir / "source.mp4"
    if existing.is_file():
        return existing
    jobstore.update(job.id, status="downloading", error=None)
    t_start_download = time.time()
    base = ["yt-dlp", "--no-playlist", "--no-progress", "--no-warnings"]
    if COOKIES_FILE and Path(COOKIES_FILE).is_file():
        base += ["--cookies", COOKIES_FILE]

    try:
        meta = json.loads(run(base + ["--skip-download", "--dump-single-json", job.url], timeout=180).stdout)
    except RuntimeError as exc:
        err_msg = str(exc).lower()
        if "403" in err_msg or "forbidden" in err_msg:
            raise RuntimeError(
                "[Descarga YouTube - HTTP 403 Forbidden] YouTube ha bloqueado temporalmente las descargas desde la IP de Kaggle. "
                "Añade tus cookies de YouTube como Kaggle Secret YT_COOKIES (formato Netscape) para autorizar la sesión o reintenta en unos minutos."
            ) from exc
        if "confirm you" in err_msg or "bot" in err_msg or "cookies" in err_msg:
            raise RuntimeError(
                "[Descarga YouTube - Bloqueo anti-bot] YouTube requiere verificación humana desde Kaggle. "
                "Añade tus cookies de YouTube como Kaggle Secret YT_COOKIES (formato Netscape) y reinicia el notebook."
            ) from exc
        raise RuntimeError(f"[Descarga YouTube fallida] {exc}") from exc

    if meta.get("is_live"):
        raise ValueError("No se pueden traducir emisiones en directo.")
    if (meta.get("duration") or 0) > MAX_DURATION_HOURS * 3600:
        raise ValueError(f"El video supera el límite de {MAX_DURATION_HOURS:g} horas.")
    jobstore.update(job.id, title=(meta.get("title") or "")[:200])

    template = str(workdir / "download.%(ext)s")
    format_selector = ["-f", "bv*[height<=720][ext=mp4]+ba[ext=m4a]/bv*[height<=720]+ba/b[height<=720]/b"]

    # Optimización: si se solicita un tramo específico (ej. 15 s), intentar descargar solo esa sección
    partial_success = False
    if job.test_start_seconds is not None or job.test_end_seconds is not None:
        start_s = float(job.test_start_seconds or 0.0)
        end_s = float(job.test_end_seconds if job.test_end_seconds is not None else start_s + 15.0)
        print(f"[autotraductor] Intentando descarga optimizada del tramo {start_s:.1f}s–{end_s:.1f}s con yt-dlp…", flush=True)
        section_cmd = (
            base
            + ["--download-sections", f"*{start_s:.2f}-{end_s:.2f}", "--force-keyframes-at-cuts"]
            + format_selector
            + ["--merge-output-format", "mp4", "--remux-video", "mp4", "-o", template, job.url]
        )
        try:
            run(section_cmd, timeout=300)
            downloaded = next((p for p in workdir.glob("download.*") if p.suffix == ".mp4"), None)
            if downloaded and downloaded.is_file() and downloaded.stat().st_size > 1024:
                try:
                    dur = probe_duration(downloaded)
                    if dur > 0.5:
                        downloaded.rename(existing)
                        partial_success = True
                        print(f"[autotraductor] Tramo descargado con éxito directamente ({dur:.1f}s, {existing.stat().st_size / 1024 / 1024:.2f} MB)", flush=True)
                except Exception:
                    pass
        except Exception as exc:
            print(f"[autotraductor] Descarga parcial no disponible para este video ({exc}); descargando video completo para recorte local.", flush=True)

    if not partial_success:
        for f in workdir.glob("download.*"):
            f.unlink(missing_ok=True)
        run(
            base
            + format_selector
            + ["--merge-output-format", "mp4", "--remux-video", "mp4", "-o", template, job.url],
            timeout=6 * 3600,
        )
        downloaded = next((p for p in workdir.glob("download.*") if p.suffix == ".mp4"), None)
        if downloaded is None:
            raise RuntimeError("yt-dlp no produjo un archivo mp4.")
        downloaded.rename(existing)

    dl_time = time.time() - t_start_download
    jobstore.add_timing(job.id, "download_seconds", dl_time)
    print(f"[autotraductor][download] Descarga completada en {dl_time:.2f}s", flush=True)
    return existing


def build_chunks(job: jobstore.Job, duration: float) -> None:
    with jobstore.job_lock(job.id):
        current = jobstore.load_job(job.id)
        if current.chunks:
            return
        if not math.isfinite(duration) or duration <= 0:
            raise ValueError("No se pudo determinar una duración válida para el video.")
        requested_start = current.test_start_seconds
        requested_end = current.test_end_seconds
        if requested_start is not None or requested_end is not None:
            start = 0.0 if requested_start is None else float(requested_start)
            end = duration if requested_end is None else float(requested_end)
            if not math.isfinite(start) or not math.isfinite(end):
                raise ValueError("El rango seleccionado debe usar tiempos válidos en segundos.")
            start = min(max(start, 0.0), duration)
            end = min(max(end, 0.0), duration)
            if end <= start:
                raise ValueError("El rango seleccionado queda fuera del video o termina antes de empezar.")
            total_range = end - start
            current.test_start_seconds = start
            current.test_end_seconds = end
            current.duration = total_range
            current.chunk_seconds = CHUNK_SECONDS
            chunk_intervals = []
            c_start = start
            while c_start < end:
                c_end = min(c_start + CHUNK_SECONDS, end)
                chunk_intervals.append((c_start, c_end))
                c_start = c_end
            current.chunks = [
                {"index": i, "start": float(cs), "end": float(ce), "status": "pending", "error": None}
                for i, (cs, ce) in enumerate(chunk_intervals)
            ]
        else:
            chunk_count = max(1, math.ceil(duration / CHUNK_SECONDS))
            starts = [i * CHUNK_SECONDS for i in range(chunk_count)]
            current.duration = duration
            current.chunk_seconds = CHUNK_SECONDS
            current.chunks = [
                {"index": i, "start": float(start), "end": float(min(start + CHUNK_SECONDS, duration)), "status": "pending", "error": None}
                for i, start in enumerate(starts)
            ]
        jobstore.save_job(current)


def process_chunk(job: jobstore.Job, index: int, source: Path) -> None:
    chunk = job.chunks[index]
    chunks_dir = job.dir / "chunks"
    chunks_dir.mkdir(exist_ok=True)
    jobstore.update_chunk(job.id, index, status="processing", error=None)
    jobstore.update(job.id, status="processing", current_chunk=index, current_chunk_progress=0.0)

    last_save = [0.0]

    def progress(fraction: float) -> None:
        if time.time() - last_save[0] > 5:
            last_save[0] = time.time()
            jobstore.update(job.id, current_chunk_progress=round(min(fraction, 1.0), 3))

    start, length = chunk["start"], chunk["end"] - chunk["start"]
    with tempfile.TemporaryDirectory(dir=job.dir) as tmp:
        tmpdir = Path(tmp)
        src_wav = tmpdir / "src.wav"
        t_ext = time.time()
        run(["ffmpeg", "-y", "-loglevel", "error", "-ss", f"{start}", "-t", f"{length}", "-i", str(source),
             "-vn", "-ac", "1", "-ar", str(SAMPLE_RATE), str(src_wav)])
        jobstore.add_timing(job.id, "mux_seconds", time.time() - t_ext)
        audio = read_wav(src_wav)
        translated, sr = translate_clone(audio, job, progress, tmpdir)

        t_mux = time.time()
        wav_tmp = tmpdir / "translated.wav"
        write_wav(wav_tmp, translated, sr)
        mp4_tmp = tmpdir / "chunk.mp4"
        mux_cmd = [
            "ffmpeg", "-y", "-loglevel", "error", "-ss", f"{start}", "-t", f"{length}", "-i", str(source),
            "-i", str(wav_tmp), "-map", "0:v:0", "-map", "1:a:0",
        ]
        if job.test_start_seconds is not None or job.test_end_seconds is not None:
            # Partial results are re-encoded for accurate range boundaries; full jobs keep stream copy.
            mux_cmd += ["-c:v", "libx264", "-preset", "ultrafast", "-crf", "24"]
        else:
            mux_cmd += ["-c:v", "copy"]
        mux_cmd += [
            "-c:a", "aac", "-b:a", "128k", "-shortest", "-movflags", "+faststart", str(mp4_tmp),
        ]
        run(mux_cmd)
        shutil.move(str(wav_tmp), chunks_dir / f"{index:03d}.wav")
        shutil.move(str(mp4_tmp), chunks_dir / f"{index:03d}.mp4")
        jobstore.add_timing(job.id, "mux_seconds", time.time() - t_mux)

    jobstore.update_chunk(job.id, index, status="completed", error=None)
    jobstore.update(job.id, current_chunk_progress=0.0)


def finalize(job: jobstore.Job, source: Path) -> None:
    workdir, chunks_dir = job.dir, job.dir / "chunks"
    result = workdir / "result.mp4"
    if not result.is_file():
        jobstore.update(job.id, status="finalizing", current_chunk=None)
        t_final = time.time()
        is_partial_range = job.test_start_seconds is not None or job.test_end_seconds is not None
        if is_partial_range:
            if len(job.chunks) == 1:
                preview_chunk = chunks_dir / "{:03d}.mp4".format(job.chunks[0]["index"])
                if not preview_chunk.is_file():
                    raise RuntimeError("No se encontró el video del rango seleccionado para previsualizarlo.")
                shutil.copy2(preview_chunk, result)
            else:
                concat_list = workdir / "concat_chunks.txt"
                mp4_paths = [chunks_dir / "{:03d}.mp4".format(c["index"]) for c in job.chunks]
                concat_list.write_text("".join("file '{}'\n".format(p.resolve()) for p in mp4_paths))
                tmp_result = workdir / "result.tmp.mp4"
                run(["ffmpeg", "-y", "-loglevel", "error", "-f", "concat", "-safe", "0", "-i", str(concat_list),
                     "-c", "copy", "-movflags", "+faststart", str(tmp_result)])
                os.replace(tmp_result, result)
                concat_list.unlink(missing_ok=True)
        else:
            concat_list = workdir / "concat.txt"
            wav_paths = [chunks_dir / "{:03d}.wav".format(c["index"]) for c in job.chunks]
            concat_list.write_text("".join("file '{}'\n".format(p) for p in wav_paths))
            audio = workdir / "audio.m4a"
            run(["ffmpeg", "-y", "-loglevel", "error", "-f", "concat", "-safe", "0", "-i", str(concat_list),
                 "-c:a", "aac", "-b:a", "160k", str(audio)])
            tmp_result = workdir / "result.tmp.mp4"
            run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(source), "-i", str(audio), "-map", "0:v:0",
                 "-map", "1:a:0", "-c:v", "copy", "-c:a", "copy", "-movflags", "+faststart", str(tmp_result)])
            os.replace(tmp_result, result)
            concat_list.unlink(missing_ok=True)
            audio.unlink(missing_ok=True)
        jobstore.add_timing(job.id, "mux_seconds", time.time() - t_final)

    source.unlink(missing_ok=True)
    for wav in chunks_dir.glob("*.wav"):
        wav.unlink(missing_ok=True)
    jobstore.update(job.id, status="completed", result_ready=True, result_bytes=result.stat().st_size,
                    current_chunk=None, error=None)

    timings = job.timings or {}
    print(
        f"[autotraductor] ✅ Trabajo {job.id} completado. Desglose de tiempos: "
        f"Descarga={timings.get('download_seconds', 0.0):.1f}s | "
        f"Whisper={timings.get('transcribe_seconds', 0.0):.1f}s | "
        f"Traducción NLLB={timings.get('translation_seconds', 0.0):.1f}s | "
        f"Síntesis voz={timings.get('tts_seconds', 0.0):.1f}s | "
        f"Ensamblado MP4={timings.get('mux_seconds', 0.0):.1f}s | "
        f"Total={job.elapsed():.1f}s.",
        flush=True,
    )


def run_job(job_id: str) -> None:
    job = jobstore.load_job(job_id)
    if job is None:
        return
    if (job.dir / "result.mp4").is_file() and job.chunks and all(c["status"] == "completed" for c in job.chunks):
        finalize(job, job.dir / "source.mp4")
        return

    source = download_source(job)
    build_chunks(job, probe_duration(source))

    job = jobstore.load_job(job_id)
    for chunk in job.chunks:
        worker.check_cancelled(job_id)
        index = chunk["index"]
        chunk_ready = (job.dir / "chunks" / f"{index:03d}.wav").is_file() and (job.dir / "chunks" / f"{index:03d}.mp4").is_file()
        if chunk["status"] == "completed" and chunk_ready:
            continue
        try:
            process_chunk(job, index, source)
        except worker.JobCancelled:
            raise
        except Exception as exc:
            message = f"{type(exc).__name__}: {exc}"[:600]
            jobstore.update_chunk(job_id, index, status="failed", error=message)
            jobstore.update(job_id, status="failed", error=f"Falló el chunk {index + 1}: {message}", current_chunk=None)
            return
        job = jobstore.load_job(job_id)

    finalize(jobstore.load_job(job_id), source)


In [ ]:
%%writefile {APP_DIR}/persistence.py
"""Optional snapshot of model caches and job state into the user's Kaggle Dataset."""

from __future__ import annotations

import json
import os
import shutil
import subprocess
import threading
import time
from pathlib import Path

from jobstore import JOBS_DIR

DATASET_SLUG = os.environ.get("AUTOTRADUCTOR_DATASET_SLUG", "autotraductor")
STAGING_DIR = Path(os.environ.get("AUTOTRADUCTOR_STAGING_DIR", "/kaggle/tmp/autotraductor-dataset"))
CACHE_DIRS = {
    "hf": Path(os.environ.get("HF_HOME", str(Path.home() / ".cache" / "huggingface"))),
    "tts": Path.home() / ".local" / "share" / "tts",
}
SKIP_NAMES = {"source.mp4", "result.tmp.mp4"}

_state = {"running": False, "last_result": None, "last_error": None, "finished_at": None}
_lock = threading.Lock()


def available() -> bool:
    return bool(os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")) and shutil.which("kaggle") is not None


def status() -> dict:
    return dict(_state)


def _copy_tree(src: Path, dst: Path) -> None:
    if not src.exists():
        return
    shutil.copytree(
        src,
        dst,
        dirs_exist_ok=True,
        ignore=lambda _dir, names: [n for n in names if n in SKIP_NAMES or n.startswith("tmp")],
    )


def _snapshot() -> None:
    try:
        shutil.rmtree(STAGING_DIR, ignore_errors=True)
        STAGING_DIR.mkdir(parents=True)
        _copy_tree(JOBS_DIR, STAGING_DIR / "jobs")
        for name, path in CACHE_DIRS.items():
            _copy_tree(path, STAGING_DIR / name)

        dataset_id = f"{os.environ['KAGGLE_USERNAME']}/{DATASET_SLUG}"
        (STAGING_DIR / "dataset-metadata.json").write_text(
            json.dumps({"title": "AutoTraductor", "id": dataset_id, "licenses": [{"name": "other"}]})
        )
        exists = subprocess.run(["kaggle", "datasets", "status", dataset_id], capture_output=True, text=True).returncode == 0
        cmd = (
            ["kaggle", "datasets", "version", "-p", str(STAGING_DIR), "-m", f"snapshot {int(time.time())}", "-r", "zip"]
            if exists
            else ["kaggle", "datasets", "create", "-p", str(STAGING_DIR), "-r", "zip"]
        )
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            raise RuntimeError((result.stderr or result.stdout).strip()[-400:])
        _state["last_result"] = "ok"
        _state["last_error"] = None
    except Exception as exc:
        _state["last_result"] = "error"
        _state["last_error"] = str(exc)[:400]
    finally:
        shutil.rmtree(STAGING_DIR, ignore_errors=True)
        _state["running"] = False
        _state["finished_at"] = time.time()


def start_snapshot() -> bool:
    with _lock:
        if _state["running"]:
            return False
        _state["running"] = True
    threading.Thread(target=_snapshot, name="autotraductor-persist", daemon=True).start()
    return True


In [ ]:
%%writefile {APP_DIR}/gradio_app.py
"""AutoTraductor — interfaz Gradio autónoma para el notebook de Kaggle (sin Cloudflare ni Vercel).

Reutiliza jobstore / worker / pipeline: este archivo solo añade la interfaz web y el arranque.
"""

from __future__ import annotations

import html
import math
import os
import re
import secrets
import shutil
import time
from pathlib import Path
from typing import Optional
from urllib.parse import quote

import gradio as gr

import jobstore
import worker
from jobstore import ACTIVE_STATUSES, DATA_DIR, JOBS_DIR, Job
from pipeline import CODE_TO_XTTS, QWEN_TTS_LANGUAGES, QWEN_TTS_MODEL_IDS

VERSION = "1.5.0-gradio"
STORAGE_LIMIT_GB = float(os.environ.get("AUTOTRADUCTOR_STORAGE_LIMIT_GB", "15"))
COOKIES_FILE = os.environ.get("AUTOTRADUCTOR_YT_COOKIES_FILE", "")
TEST_RANGE_CHOICES = [
    ("Primeros 15 segundos (00:00:15) — prueba rápida recomendada", "first15"),
    ("Primeros 10 segundos (00:00:10)", "first10"),
    ("Tramo personalizado (con reloj y selectores)", "custom"),
    ("Video completo", "full"),
]
MAX_QUEUED_JOBS = int(os.environ.get("AUTOTRADUCTOR_MAX_QUEUED_JOBS", "5"))
STARTED_AT = time.time()

YOUTUBE_URL_RE = re.compile(
    r"^https?://(www\.|m\.|music\.)?(youtube\.com/(watch\?v=|shorts/|live/|embed/)|youtu\.be/)[A-Za-z0-9_\-]{6,}",
)
JOB_ID_RE = re.compile(r"^[a-f0-9]{16}$")
AUTO = "auto"

# (código, nombre, admitido en Whisper, admitido en XTTS v2)
LANGUAGES = [
    ("eng", "Inglés", True, True), ("spa", "Español", True, True), ("por", "Portugués", True, True),
    ("fra", "Francés", True, True), ("deu", "Alemán", True, True), ("ita", "Italiano", True, True),
    ("nld", "Neerlandés", True, True), ("pol", "Polaco", True, True), ("rus", "Ruso", True, True),
    ("ukr", "Ucraniano", True, False), ("tur", "Turco", True, True), ("ces", "Checo", True, True),
    ("hun", "Húngaro", True, True), ("ron", "Rumano", True, False), ("swe", "Sueco", True, False),
    ("dan", "Danés", True, False), ("fin", "Finés", True, False), ("cat", "Catalán", True, False),
    ("ell", "Griego", True, False), ("heb", "Hebreo", True, False), ("arb", "Árabe", True, True),
    ("pes", "Persa", True, False), ("hin", "Hindi", True, True), ("urd", "Urdu", True, False),
    ("ben", "Bengalí", True, False), ("cmn", "Chino mandarín", True, True), ("jpn", "Japonés", True, True),
    ("kor", "Coreano", True, True), ("vie", "Vietnamita", True, False), ("tha", "Tailandés", True, False),
    ("ind", "Indonesio", True, False), ("tgl", "Tagalo", True, False), ("swh", "Suajili", True, False),
]
if {code for code, _, _, xtts_supported in LANGUAGES if xtts_supported} != set(CODE_TO_XTTS):
    raise RuntimeError("Los idiomas de clonación de la interfaz deben coincidir con CODE_TO_XTTS")
LANG_NAMES = {code: name for code, name, _, _ in LANGUAGES}

STATUS_LABEL = {
    "queued": "En cola", "downloading": "Descargando video", "processing": "Traduciendo",
    "finalizing": "Uniendo partes", "completed": "Completado", "failed": "Fallido",
}
CHUNK_LABEL = {"pending": "Pendiente", "processing": "Procesando", "completed": "Completado", "failed": "Fallido"}
TTS_ENGINE_LABEL = {
    "xtts": "XTTS v2 · opción predeterminada de clonación (17 idiomas)",
    "qwen06": "Qwen3-TTS Base 0.6B · alternativa de clonación (10 idiomas)",
}
QWEN_TTS_ENGINES = set(QWEN_TTS_MODEL_IDS)
CLONE_INSTALL_LOG = Path(os.environ.get("AUTOTRADUCTOR_CLONE_INSTALL_LOG", "/kaggle/tmp/clone_install.log"))
CLONE_INSTALL_HELP = (
    "Revisa el registro completo en `/kaggle/tmp/clone_install.log` y vuelve a ejecutar la celda de dependencias; "
    "confirma que imprime «Modo clonación: OK». Si instalaste los paquetes después de arrancar Gradio, "
    "vuelve a ejecutar la última celda para reiniciar la app."
)


# --------------------------------------------------------------------------- utilidades

def source_choices() -> list[tuple[str, str]]:
    return [("Detectar automáticamente", AUTO)] + [(name, code) for code, name, _, _ in LANGUAGES]


def target_choices(tts_engine: str = "xtts") -> list[tuple[str, str]]:
    supported = set(QWEN_TTS_LANGUAGES) if tts_engine in QWEN_TTS_ENGINES else set(CODE_TO_XTTS)
    return [(name, code) for code, name, _, _ in LANGUAGES if code in supported]


def clone_install_log_excerpt(limit: int = 900) -> Optional[str]:
    try:
        with CLONE_INSTALL_LOG.open("rb") as log:
            log.seek(0, os.SEEK_END)
            size = log.tell()
            log.seek(max(0, size - 8192))
            tail = log.read().decode("utf-8", errors="replace")
    except OSError:
        return None

    lines = [line.strip() for line in tail.splitlines() if line.strip()]
    relevant = [
        line for line in lines
        if re.search(r"error|traceback|importerror|modulenotfounderror|runtimeerror|failed|conflict|not found", line, re.I)
        and "dependency resolver does not currently take into account" not in line.lower()
    ]
    excerpt = " | ".join((relevant[-5:] or lines[-5:]))
    return excerpt[-limit:] or None


def clone_unavailable_message(reason: Optional[str] = None) -> str:
    available, detected_reason = worker.clone_mode_status("xtts")
    if available:
        return ""

    details = reason or detected_reason or "no se pudo importar XTTS v2 y faster-whisper"
    message = f"Clonación XTTS no disponible. Motivo: {details}."
    log_excerpt = clone_install_log_excerpt()
    if log_excerpt:
        message += f" Detalle de {CLONE_INSTALL_LOG}: {log_excerpt}."
    return f"{message} {CLONE_INSTALL_HELP}"


def _markdown_text(value: str) -> str:
    return html.escape(value, quote=False).replace("`", "&#96;").replace("\n", " ")


def fmt_duration(seconds: float) -> str:
    total = max(0, int(round(seconds)))
    h, m, s = total // 3600, (total % 3600) // 60, total % 60
    return f"{h}:{m:02d}:{s:02d}" if h else f"{m}:{s:02d}"


def fmt_bytes(size: float) -> str:
    if size < 1024**2:
        return f"{size / 1024:.0f} KB"
    if size < 1024**3:
        return f"{size / 1024**2:.1f} MB"
    return f"{size / 1024**3:.2f} GB"


def storage_usage() -> dict:
    data_dir = jobstore.DATA_DIR
    try:
        data_dir.mkdir(parents=True, exist_ok=True)
    except OSError:
        pass
    used = 0
    if data_dir.exists():
        for path in data_dir.rglob("*"):
            try:
                if path.is_file():
                    used += path.stat().st_size
            except OSError:
                continue
    try:
        disk = shutil.disk_usage(data_dir if data_dir.exists() else "/")
        free = disk.free
    except OSError:
        free = int(STORAGE_LIMIT_GB * 1024**3)
    limit = int(STORAGE_LIMIT_GB * 1024**3)
    return {"used": used, "limit": limit, "free": free, "full": used >= limit or free < 2 * 1024**3}


def gpu_name() -> Optional[str]:
    """Show every CUDA device made available by Kaggle, not only device 0."""
    try:
        import torch

        if not torch.cuda.is_available():
            return None
        names = [torch.cuda.get_device_name(index) for index in range(torch.cuda.device_count())]
        if not names:
            return None
        if len(names) > 1:
            return f"{names[0]} × {len(names)} (Multi-GPU activo)"
        return names[0]
    except Exception:
        return None


def file_url(path: Path) -> str:
    """Enlace directo a un archivo dentro de DATA_DIR."""
    return "/gradio_api/file=" + quote(str(path.resolve()), safe="/")


def format_timestamp(seconds: float) -> str:
    s = max(0.0, float(seconds))
    h = int(s // 3600)
    m = int((s % 3600) // 60)
    sec = int(s % 60)
    return f"{h:02d}:{m:02d}:{sec:02d}"


def format_duration(seconds: float) -> str:
    s = max(0.0, float(seconds))
    h = int(s // 3600)
    m = int((s % 3600) // 60)
    sec = int(s % 60)
    if h > 0:
        return f"{h}h {m:02d}m {sec:02d}s ({int(s)} segundos en total)"
    elif m > 0:
        return f"{m}m {sec:02d}s ({int(s)} segundos en total)"
    else:
        return f"{sec}s"


def parse_timestamp(val) -> float:
    if isinstance(val, (int, float)):
        return max(0.0, float(val))
    val = str(val).strip()
    if not val:
        return 0.0
    if ":" in val:
        parts = val.split(":")
        if len(parts) == 2:
            m, s = float(parts[0]), float(parts[1])
            return m * 60.0 + s
        elif len(parts) == 3:
            h, m, s = float(parts[0]), float(parts[1]), float(parts[2])
            return h * 3600.0 + m * 60.0 + s
    val = val.rstrip("sS")
    return float(val)


def probe_youtube_metadata(url: str) -> tuple[Optional[float], Optional[str]]:
    url = (url or "").strip()
    if not YOUTUBE_URL_RE.match(url):
        return None, None
    base = ["yt-dlp", "--no-playlist", "--no-warnings"]
    if COOKIES_FILE and Path(COOKIES_FILE).is_file():
        base += ["--cookies", COOKIES_FILE]
    # Intento 1: extracción directa ultrarrápida de solo duración y título con --print
    try:
        fast_cmd = base + ["--print", "%(duration)s", "--print", "%(title)s", url]
        proc = subprocess.run(fast_cmd, capture_output=True, text=True, timeout=12)
        if proc.returncode == 0 and proc.stdout.strip():
            lines = [l.strip() for l in proc.stdout.strip().splitlines() if l.strip()]
            if lines:
                try:
                    dur = float(lines[0])
                    title = lines[1] if len(lines) > 1 else None
                    if dur > 0:
                        return dur, title
                except (ValueError, IndexError):
                    pass
    except Exception:
        pass

    # Intento 2: fallback a dump-single-json
    try:
        proc = subprocess.run(base + ["--skip-download", "--dump-single-json", url], capture_output=True, text=True, timeout=20)
        if proc.returncode == 0 and proc.stdout.strip():
            meta = json.loads(proc.stdout)
            dur = meta.get("duration")
            title = meta.get("title")
            return (float(dur) if dur is not None and dur > 0 else None, str(title) if title else None)
    except Exception:
        pass
    return None, None


# --------------------------------------------------------------------------- acciones

def resolve_test_range(scope: str, start_value, end_value) -> tuple[Optional[float], Optional[float]]:
    """Convert the Gradio range selector into an optional absolute source interval."""
    if scope == "full":
        return None, None
    if scope == "first10":
        return 0.0, 10.0
    if scope == "first15":
        return 0.0, 15.0
    if scope != "custom":
        raise ValueError("Selecciona un tramo válido.")

    try:
        start = parse_timestamp(start_value)
        end = parse_timestamp(end_value)
    except (TypeError, ValueError):
        raise ValueError("Escribe el inicio y el final del rango en formato HH:MM:SS, MM:SS o segundos.") from None
    if not math.isfinite(start) or not math.isfinite(end):
        raise ValueError("Los tiempos del rango deben ser números válidos.")
    if start < 0 or end <= start:
        raise ValueError("El final del rango debe ser mayor que el inicio y ambos deben ser positivos.")
    if end - start < 1.0:
        raise ValueError("El rango debe durar al menos 1 segundo.")
    return start, end


def create_job(
    url: str,
    source: str,
    target: str,
    tts_engine: str = "xtts",
    test_scope: str = "first15",
    test_start_value=0,
    test_end_value=15,
) -> Optional[str]:
    url = (url or "").strip()
    if not YOUTUBE_URL_RE.match(url):
        gr.Warning("Pega un enlace de YouTube válido (youtube.com/watch?v=… o youtu.be/…).")
        return None
    if tts_engine not in TTS_ENGINE_LABEL:
        gr.Warning("Selecciona un motor de voz válido.")
        return None
    if target not in {value for _, value in target_choices(tts_engine)}:
        gr.Warning("El idioma destino no está disponible en el motor de voz seleccionado.")
        return None
    if source != AUTO and source not in LANG_NAMES:
        gr.Warning("Idioma de origen no válido.")
        return None
    try:
        test_start, test_end = resolve_test_range(test_scope, test_start_value, test_end_value)
    except ValueError as exc:
        gr.Warning(str(exc))
        return None
    if storage_usage()["full"]:
        gr.Warning("Espacio lleno. Descarga y borra trabajos terminados antes de iniciar uno nuevo.")
        return None

    available, reason = worker.mode_status("clone", tts_engine)
    if not available:
        if tts_engine in QWEN_TTS_ENGINES:
            gr.Warning(f"Qwen3-TTS no está disponible. Motivo: {reason or 'revisa la celda de instalación de Qwen'}.")
        else:
            gr.Warning(clone_unavailable_message(reason))
        return None

    pending = [j for j in jobstore.list_jobs() if j.status in ACTIVE_STATUSES]
    if len(pending) >= MAX_QUEUED_JOBS:
        gr.Warning(f"Ya hay {len(pending)} trabajos en cola. Espera a que terminen.")
        return None

    job = Job.new(
        job_id=secrets.token_hex(8), url=url, source_lang=source, target_lang=target, mode="clone", tts_engine=tts_engine
    )
    job.test_start_seconds = test_start
    job.test_end_seconds = test_end
    jobstore.save_job(job)
    worker.enqueue(job.id)
    gr.Info("Trabajo creado y añadido a la cola.")
    return job.id


def retry_job(job_id: Optional[str]) -> None:
    if not job_id or not JOB_ID_RE.match(job_id):
        gr.Warning("Selecciona un trabajo.")
        return
    with jobstore.job_lock(job_id):
        job = jobstore.load_job(job_id)
        if job is None:
            gr.Warning("Trabajo no encontrado.")
            return
        if job.status != "failed":
            gr.Warning("Solo se pueden reintentar trabajos fallidos.")
            return
        job.status = "queued"
        job.error = None
        for chunk in job.chunks:
            if chunk["status"] in ("failed", "processing"):
                chunk["status"] = "pending"
                chunk["error"] = None
        jobstore.save_job(job)
    worker.enqueue(job_id)
    gr.Info("Reintentando el trabajo en cola.")


def delete_job(job_id: Optional[str], confirmed: bool) -> None:
    if not job_id or not JOB_ID_RE.match(job_id):
        gr.Warning("Selecciona un trabajo.")
        return
    if not confirmed:
        gr.Warning("Marca «Confirmar borrado» para borrar el trabajo y todos sus archivos.")
        return
    if jobstore.load_job(job_id) is None:
        gr.Warning("Trabajo no encontrado.")
        return
    if worker.current_job_id() == job_id:
        worker.cancel(job_id)
        for _ in range(60):
            if worker.current_job_id() != job_id:
                break
            time.sleep(0.5)
    shutil.rmtree(JOBS_DIR / job_id, ignore_errors=True)
    gr.Info("Trabajo borrado.")


def persist_snapshot() -> None:
    import persistence

    if not persistence.available():
        gr.Warning("Faltan los secrets KAGGLE_USERNAME / KAGGLE_KEY para guardar en el Dataset.")
        return
    started = persistence.start_snapshot()
    gr.Info("Guardando en el Dataset de Kaggle…" if started else "Ya hay un guardado en curso.")


# --------------------------------------------------------------------------- vistas

CSS = """
.at-job { border: 1px solid var(--border-color-primary); border-radius: 12px; padding: 14px 16px; margin-bottom: 12px;
          background: var(--background-fill-secondary); }
.at-head { display: flex; justify-content: space-between; gap: 12px; align-items: flex-start; flex-wrap: wrap; }
.at-title { font-weight: 600; word-break: break-word; }
.at-meta { font-size: 12px; color: var(--body-text-color-subdued); margin-top: 2px; }
.at-badge { font-size: 12px; padding: 2px 10px; border-radius: 999px; border: 1px solid var(--border-color-primary); white-space: nowrap; }
.at-badge.ok { background: #16a34a22; color: #16a34a; border-color: #16a34a66; }
.at-badge.bad { background: #dc262622; color: #dc2626; border-color: #dc262666; }
.at-badge.run { background: #2563eb22; color: #3b82f6; border-color: #3b82f666; }
.at-badge.queue { background: #f59e0b22; color: #d97706; border-color: #d9770666; }
.at-bar { height: 8px; border-radius: 999px; background: var(--border-color-primary); overflow: hidden; margin: 10px 0 4px; }
.at-bar > span { display: block; height: 100%; background: var(--color-accent); transition: width .4s; }
.at-row { display: flex; justify-content: space-between; font-size: 12px; color: var(--body-text-color-subdued); gap: 8px; flex-wrap: wrap; margin-top: 3px; }
.at-timings { font-size: 12px; color: var(--color-accent); margin-top: 5px; font-weight: 500; }
.at-error { margin-top: 8px; padding: 8px 10px; border-radius: 8px; background: #dc262614; color: #dc2626; font-size: 13px; }
.at-dl { display: inline-block; margin-top: 10px; padding: 6px 14px; border-radius: 8px; background: var(--color-accent);
         color: white !important; text-decoration: none; font-size: 14px; font-weight: 500; }
.at-empty { text-align: center; padding: 28px; border: 1px dashed var(--border-color-primary); border-radius: 12px;
            color: var(--body-text-color-subdued); }
.at-chunks { width: 100%; border-collapse: collapse; font-size: 13px; }
.at-chunks td, .at-chunks th { padding: 6px 8px; border-bottom: 1px solid var(--border-color-primary); text-align: left; }
"""


def render_jobs(jobs: list[Job]) -> str:
    if not jobs:
        return '<div class="at-empty">Aún no hay trabajos. Pega un enlace de YouTube para empezar.</div>'
    cards = []
    for job in jobs:
        data = job.public()
        status = job.status
        badge_cls = "ok" if status == "completed" else "bad" if status == "failed" else "queue" if status == "queued" else "run"
        title = html.escape(job.title or job.url)
        src = "Auto" if job.source_lang == AUTO else LANG_NAMES.get(job.source_lang, job.source_lang)
        engine_label = TTS_ENGINE_LABEL.get(job.tts_engine, job.tts_engine)
        meta = f"{src} → {LANG_NAMES.get(job.target_lang, job.target_lang)} · {html.escape(engine_label)}"
        if job.duration > 0:
            meta += f" · {fmt_duration(job.duration)}"
        if job.test_start_seconds is not None or job.test_end_seconds is not None:
            test_start = job.test_start_seconds or 0.0
            test_end = job.test_end_seconds if job.test_end_seconds is not None else job.duration
            meta += f" · Tramo {fmt_duration(test_start)}–{fmt_duration(test_end)}"
        meta += f" · id {job.id}"

        body = ""
        if status != "completed":
            current = job.current_chunk + 1 if job.current_chunk is not None else data["chunks_completed"]
            if job.test_start_seconds is not None or job.test_end_seconds is not None:
                label = "Tramo seleccionado"
            elif data["chunks_total"] == 1:
                label = "Vídeo completo (un solo bloque)"
            else:
                label = f"Bloque {max(current, 1)} de {data['chunks_total']}" if data["chunks_total"] else "Preparando…"
            body += (
                f'<div class="at-bar"><span style="width:{data["progress"]:.1f}%"></span></div>'
                f'<div class="at-row"><span>{label}</span>'
                f'<span>Progreso estimado del audio: {fmt_duration(data["processed_seconds"])} / '
                f'{fmt_duration(job.duration)} · {data["progress"]:.1f}%</span></div>'
            )
        elapsed = fmt_duration(data["elapsed_seconds"])
        if status == "queued":
            pos = data.get("queue_position") or 1
            tot = data.get("queue_total") or 1
            clock_label = f"En cola · Turno #{pos} de {tot} en espera · ejecución previa: {elapsed}"
            status_text = f"En cola (#{pos}/{tot})"
        elif status == "completed":
            clock_label = f"Tiempo total de ejecución: {elapsed}"
            status_text = "Completado"
        else:
            clock_label = f"Tiempo transcurrido en GPU: {elapsed}"
            status_text = STATUS_LABEL.get(status, status)
        body += f'<div class="at-row"><span>{html.escape(clock_label)}</span></div>'

        # Desglose de tiempos por etapa
        timings = data.get("timings") or {}
        has_timings = any(float(v) > 0 for v in timings.values())
        if has_timings:
            t_parts = []
            if timings.get("download_seconds", 0) > 0:
                t_parts.append(f"Descarga: {timings['download_seconds']:.1f}s")
            if timings.get("transcribe_seconds", 0) > 0:
                t_parts.append(f"Whisper: {timings['transcribe_seconds']:.1f}s")
            if timings.get("translation_seconds", 0) > 0:
                t_parts.append(f"NLLB-200: {timings['translation_seconds']:.1f}s")
            if timings.get("tts_seconds", 0) > 0:
                engine_short = "XTTS" if job.tts_engine == "xtts" else "Qwen"
                t_parts.append(f"Voz {engine_short}: {timings['tts_seconds']:.1f}s")
            if timings.get("mux_seconds", 0) > 0:
                t_parts.append(f"Ensamblado MP4: {timings['mux_seconds']:.1f}s")
            body += f'<div class="at-timings">⏱️ {html.escape(" · ".join(t_parts))}</div>'

        if job.error:
            body += f'<div class="at-error"><b>Error:</b> {html.escape(job.error)}</div>'
        result = job.dir / "result.mp4"
        if job.result_ready and result.is_file():
            body += (
                f'<a class="at-dl" href="{file_url(result)}" download="autotraductor-{job.id}.mp4" target="_blank">'
                f"⬇ Descargar MP4 ({fmt_bytes(job.result_bytes)})</a>"
            )
        cards.append(
            f'<div class="at-job"><div class="at-head"><div><div class="at-title">{title}</div>'
            f'<div class="at-meta">{html.escape(meta)}</div></div>'
            f'<span class="at-badge {badge_cls}">{html.escape(status_text)}</span></div>{body}</div>'
        )
    return "".join(cards)


def render_detail(job_id: Optional[str]) -> str:
    job = jobstore.load_job(job_id) if job_id and JOB_ID_RE.match(job_id) else None
    if job is None:
        return '<div class="at-empty">Selecciona un trabajo para ver sus partes.</div>'
    if not job.chunks:
        return '<div class="at-empty">Este trabajo aún no tiene partes (se crean tras descargar el video).</div>'
    rows = []
    single_block = len(job.chunks) == 1
    test_range = job.test_start_seconds is not None or job.test_end_seconds is not None
    for chunk in job.chunks:
        path = job.dir / "chunks" / f"{chunk['index']:03d}.mp4"
        link = ""
        if chunk["status"] == "completed" and path.is_file():
            name = f"autotraductor-{job.id}-parte-{chunk['index'] + 1:03d}.mp4"
            link = f'<a href="{file_url(path)}" download="{name}" target="_blank">⬇ Descargar</a>'
        error = f"<br><small>{html.escape(chunk['error'])}</small>" if chunk.get("error") else ""
        label = "Tramo seleccionado" if test_range else "Vídeo completo" if single_block else f"#{chunk['index'] + 1:03d}"
        rows.append(
            f"<tr><td>{label}</td><td>{fmt_duration(chunk['start'])}–{fmt_duration(chunk['end'])}</td>"
            f"<td>{CHUNK_LABEL.get(chunk['status'], chunk['status'])}{error}</td><td>{link}</td></tr>"
        )
    return (
        '<table class="at-chunks"><thead><tr><th>Bloque</th><th>Tiempo</th><th>Estado</th><th></th></tr></thead><tbody>'
        + "".join(rows)
        + "</tbody></table>"
    )


def render_preview(job_id: Optional[str]) -> Optional[str]:
    job = jobstore.load_job(job_id) if job_id and JOB_ID_RE.match(job_id) else None
    if job is None or not job.result_ready:
        return None
    result = job.dir / "result.mp4"
    return str(result.resolve()) if result.is_file() else None


def render_status() -> str:
    storage = storage_usage()
    gpu = gpu_name()
    xtts_available, xtts_reason = worker.clone_mode_status("xtts")
    qwen_available, qwen_reason = worker.qwen_tts_status()
    percent = min(100, 100 * storage["used"] / storage["limit"]) if storage["limit"] else 0
    lines = [
        f"**GPU(s) visibles:** {gpu if gpu else '⚠️ sin GPU (activa Settings → Accelerator → GPU)'}  ",
        f"**Worker:** {'activo (1 trabajo en GPU a la vez; cola secuencial segura)' if worker.is_alive() else '⚠️ detenido'} · "
        f"**Traducción texto:** NLLB-200 distilled 600M listo · "
        f"**XTTS v2 (predeterminado):** {'disponible' if xtts_available else 'no disponible'} · "
        f"**Qwen3-TTS 0.6B (alternativa):** {'disponible' if qwen_available else 'no disponible'} · "
        f"**Versión:** {VERSION}  ",
        f"**Espacio usado:** {fmt_bytes(storage['used'])} / {fmt_bytes(storage['limit'])} ({percent:.0f} %) · "
        f"libre en disco: {fmt_bytes(storage['free'])}",
    ]
    if not xtts_available:
        lines.extend([
            "",
            f"⚠️ **XTTS v2 no disponible. Motivo:** `{_markdown_text(xtts_reason or 'dependencias no detectadas')}`.",
        ])
        log_excerpt = clone_install_log_excerpt(limit=650)
        if log_excerpt:
            lines.append(f"**Último detalle de `{CLONE_INSTALL_LOG}`:** `{_markdown_text(log_excerpt)}`.")
        lines.append(f"**Cómo resolverlo:** {CLONE_INSTALL_HELP}")
    if os.environ.get("AUTOTRADUCTOR_QWEN_TTS_READY") == "1" and not qwen_available:
        lines.extend([
            "",
            f"⚠️ **Qwen3-TTS 0.6B no disponible. Motivo:** `{_markdown_text(qwen_reason or 'falló la comprobación de importación')}`.",
        ])
    if storage["full"]:
        lines.append("\n⚠️ **Espacio lleno:** no se aceptan nuevos trabajos. Descarga y borra trabajos terminados.")
    return "\n".join(lines)


def job_choices(jobs: list[Job]) -> list[tuple[str, str]]:
    return [(f"{(j.title or j.url)[:70]} · {STATUS_LABEL.get(j.status, j.status)} · {j.id}", j.id) for j in jobs]


def refresh(selected: Optional[str], last_choices: list, last_preview: Optional[str]):
    """Refresh jobs while avoiding reloading a video preview on every timer tick."""
    jobs = jobstore.list_jobs()
    choices = job_choices(jobs)
    if choices == last_choices:
        selector = gr.update()
        effective_selection = selected
    else:
        valid = {value for _, value in choices}
        effective_selection = selected if selected in valid else (choices[0][1] if choices else None)
        selector = gr.update(choices=choices, value=effective_selection)

    preview = render_preview(effective_selection)
    preview_update = gr.update() if preview == last_preview else gr.update(value=preview)
    return (
        render_jobs(jobs), selector, choices, render_detail(effective_selection), preview_update, preview,
    )


def render_selected(job_id: Optional[str], last_preview: Optional[str]):
    preview = render_preview(job_id)
    preview_update = gr.update() if preview == last_preview else gr.update(value=preview)
    return render_detail(job_id), preview_update, preview


# --------------------------------------------------------------------------- interfaz

def build_ui() -> gr.Blocks:
    with gr.Blocks(title="AutoTraductor", theme=gr.themes.Soft(), css=CSS) as demo:
        gr.Markdown(
            "# 🎙️ AutoTraductor\n"
            "Traduce y dobla videos de YouTube con IA. El pipeline combina **Whisper** para transcripción, "
            "**NLLB-200 distilled 600M** para traducción de texto multilingüe y **XTTS v2 / Qwen3-TTS** para clonación de voz. "
            "Aprovecha automáticamente **2 GPU T4** en paralelo cuando están disponibles. "
            "**Mantén el notebook abierto** mientras se procesa."
        )
        status_md = gr.Markdown(render_status())
        qwen_available, _ = worker.qwen_tts_status()
        tts_engine_choices = [(TTS_ENGINE_LABEL["xtts"], "xtts")]
        if qwen_available:
            tts_engine_choices.append((TTS_ENGINE_LABEL["qwen06"], "qwen06"))

        default_tts_engine = "xtts"  # XTTS v2 como opción predeterminada
        choices_state = gr.State([])
        preview_state = gr.State(None)

        with gr.Row(equal_height=False):
            with gr.Column(scale=5):
                gr.Markdown("### Nueva traducción")
                with gr.Row():
                    url = gr.Textbox(
                        label="Enlace de YouTube",
                        placeholder="https://www.youtube.com/watch?v=…",
                        scale=4,
                    )
                    detect_btn = gr.Button("🔍 Detectar duración", scale=1, size="sm")
                url_meta_md = gr.Markdown(visible=False)
                tts_engine = gr.Radio(
                    choices=tts_engine_choices,
                    value=default_tts_engine,
                    label="Motor de voz para clonación",
                    info="XTTS v2 es la opción predeterminada (17 idiomas). Qwen3-TTS Base 0.6B es la alternativa rápida (10 idiomas).",
                )
                with gr.Row():
                    source = gr.Dropdown(choices=source_choices(), value=AUTO, label="Idioma original")
                    target = gr.Dropdown(choices=target_choices("xtts"), value="spa", label="Traducir a")
                gr.Markdown("#### Tramo del video que quieres procesar")
                test_scope = gr.Radio(
                    choices=TEST_RANGE_CHOICES,
                    value="first15",
                    label="Tramo a procesar (el MP4 contendrá solo este tramo)",
                )
                with gr.Column(visible=False) as custom_range_col:
                    gr.Markdown("##### Ajusta el tramo con los selectores o escribe el tiempo en HH:MM:SS (sin límite de duración)")
                    with gr.Row():
                        test_start_str = gr.Textbox(
                            label="Inicio (HH:MM:SS)",
                            value="00:00:00",
                            scale=1,
                            info="Ej: 00:00:00",
                        )
                        test_end_str = gr.Textbox(
                            label="Fin (HH:MM:SS)",
                            value="00:00:15",
                            scale=1,
                            info="⚠️ Ej: 00:00:15 = 15 segundos. (¡15:00 significa 15 minutos!)",
                        )
                    with gr.Row():
                        test_start_slider = gr.Slider(minimum=0, maximum=86400, value=0, step=1, label="Selector de Inicio", scale=1)
                        test_end_slider = gr.Slider(minimum=0, maximum=86400, value=15, step=1, label="Selector de Fin", scale=1)
                    range_summary = gr.Markdown("📏 **Tramo:** 00:00:00 a 00:00:15 (Duración: 15s)")

                test_scope.change(
                    lambda scope: gr.update(visible=scope == "custom"),
                    inputs=test_scope,
                    outputs=custom_range_col,
                )
                gr.Markdown(
                    "Por defecto procesa los primeros 15 s (recomendado para verificar voz y entonación rápidamente). "
                    "También puedes elegir 10 s, un tramo personalizado (sin límite de duración) o el video completo. "
                    "Para tramos específicos, se intenta la descarga directa de la sección con yt-dlp para ahorrar tiempo y ancho de banda."
                )
                start = gr.Button("Iniciar traducción", variant="primary")
                gr.Markdown("El audio original se silencia y se sustituye por completo por el traducido.")

                gr.Markdown("### Trabajo seleccionado")
                selector = gr.Dropdown(choices=[], value=None, label="Trabajo", interactive=True)
                with gr.Row():
                    retry = gr.Button("Reintentar (si falló)")
                    confirm = gr.Checkbox(label="Confirmar borrado", value=False)
                    delete = gr.Button("Borrar trabajo", variant="stop")
                persist = gr.Button("Guardar modelos y estado en Dataset (opcional)", size="sm")

            with gr.Column(scale=7):
                gr.Markdown("### Trabajos en cola y finalizados")
                jobs_html = gr.HTML()
                gr.Markdown("### Bloques del trabajo seleccionado")
                detail_html = gr.HTML()
                gr.Markdown("### Vista previa (antes de descargar)")
                preview_video = gr.Video(label="El video aparecerá aquí al terminar", interactive=False)

        timer = gr.Timer(4)
        slow_timer = gr.Timer(15)

        refresh_in = [selector, choices_state, preview_state]
        refresh_out = [jobs_html, selector, choices_state, detail_html, preview_video, preview_state]

        def on_tts_engine(engine: str, current_tgt: str):
            choices = target_choices(engine)
            allowed = {value for _, value in choices}
            return gr.update(choices=choices, value=current_tgt if current_tgt in allowed else choices[0][1])

        def on_detect_video(u: str):
            dur, title = probe_youtube_metadata(u)
            if dur is not None and dur > 0:
                max_val = max(float(dur) * 1.05, 86400.0)
                formatted_dur = format_timestamp(dur)
                meta_text = f"📺 **Video:** {title or 'YouTube'} · ⏱️ **Duración:** `{formatted_dur}` ({int(dur)}s)"
                summary = f"📏 **Tramo:** 00:00:00 a {formatted_dur} (Duración: {format_duration(dur)})"
                return (
                    gr.update(maximum=max_val, value=0),
                    gr.update(maximum=max_val, value=dur),
                    gr.update(value="00:00:00"),
                    gr.update(value=formatted_dur),
                    gr.update(value=summary),
                    gr.update(value=meta_text, visible=True),
                )
            return (
                gr.update(maximum=86400.0),
                gr.update(maximum=86400.0),
                gr.update(),
                gr.update(),
                gr.update(),
                gr.update(value="ℹ️ Introduce el tramo que quieras procesar arriba (ej. 02:39:50 o en segundos).", visible=True),
            )

        def on_start_slider(s_val, e_val):
            s_s = min(float(s_val), float(e_val))
            dur = max(0.0, float(e_val) - s_s)
            t_str = format_timestamp(s_s)
            summary = f"📏 **Tramo:** {t_str} a {format_timestamp(e_val)} (Duración: {format_duration(dur)})"
            return t_str, summary

        def on_end_slider(s_val, e_val):
            e_s = max(float(s_val), float(e_val))
            dur = max(0.0, e_s - float(s_val))
            t_str = format_timestamp(e_s)
            summary = f"📏 **Tramo:** {format_timestamp(s_val)} a {t_str} (Duración: {format_duration(dur)})"
            return t_str, summary

        def on_start_str_edit(txt, current_end):
            try:
                val = parse_timestamp(txt)
                new_max = max(val * 1.25, 86400.0)
                dur = max(0.0, float(current_end) - val)
                summary = f"📏 **Tramo:** {format_timestamp(val)} a {format_timestamp(current_end)} (Duración: {format_duration(dur)})"
                return gr.update(maximum=new_max, value=val), format_timestamp(val), summary
            except Exception:
                return gr.update(), txt, gr.update()

        def on_end_str_edit(txt, current_start):
            try:
                val = parse_timestamp(txt)
                new_max = max(val * 1.25, 86400.0)
                dur = max(0.0, val - float(current_start))
                summary = f"📏 **Tramo:** {format_timestamp(current_start)} a {format_timestamp(val)} (Duración: {format_duration(dur)})"
                return gr.update(maximum=new_max, value=val), format_timestamp(val), summary
            except Exception:
                return gr.update(), txt, gr.update()

        def on_start(u, s, t, engine, scope, range_start, range_end, sel, last, last_preview):
            created_job_id = create_job(u, s, t, engine, scope, range_start, range_end)
            if created_job_id:
                sel = created_job_id
            return ("" if created_job_id else u, *refresh(sel, last, last_preview))

        def on_retry(sel, last, last_preview):
            retry_job(sel)
            return refresh(sel, last, last_preview)

        def on_delete(sel, ok, last, last_preview):
            delete_job(sel, ok)
            return (False, *refresh(sel, last, last_preview))

        def on_persist(sel, last, last_preview):
            persist_snapshot()
            return refresh(sel, last, last_preview)

        detect_btn.click(
            on_detect_video,
            [url],
            [test_start_slider, test_end_slider, test_start_str, test_end_str, range_summary, url_meta_md],
        )
        url.submit(
            on_detect_video,
            [url],
            [test_start_slider, test_end_slider, test_start_str, test_end_str, range_summary, url_meta_md],
        )
        test_start_slider.change(
            on_start_slider,
            [test_start_slider, test_end_slider],
            [test_start_str, range_summary],
        )
        test_end_slider.change(
            on_end_slider,
            [test_start_slider, test_end_slider],
            [test_end_str, range_summary],
        )
        test_start_str.blur(
            on_start_str_edit,
            [test_start_str, test_end_slider],
            [test_start_slider, test_start_str, range_summary],
        )
        test_start_str.submit(
            on_start_str_edit,
            [test_start_str, test_end_slider],
            [test_start_slider, test_start_str, range_summary],
        )
        test_end_str.blur(
            on_end_str_edit,
            [test_end_str, test_start_slider],
            [test_end_slider, test_end_str, range_summary],
        )
        test_end_str.submit(
            on_end_str_edit,
            [test_end_str, test_start_slider],
            [test_end_slider, test_end_str, range_summary],
        )

        tts_engine.change(on_tts_engine, [tts_engine, target], target)
        start.click(
            on_start,
            [url, source, target, tts_engine, test_scope, test_start_str, test_end_str, *refresh_in],
            [url, *refresh_out],
        )
        retry.click(on_retry, refresh_in, refresh_out)
        delete.click(on_delete, [selector, confirm, choices_state, preview_state], [confirm, *refresh_out])
        persist.click(on_persist, refresh_in, refresh_out)
        selector.change(
            render_selected,
            [selector, preview_state],
            [detail_html, preview_video, preview_state],
        )
        timer.tick(refresh, refresh_in, refresh_out)
        slow_timer.tick(render_status, None, status_md)
        demo.load(refresh, refresh_in, refresh_out)
    return demo


def main() -> None:
    jobstore.ensure_dirs()
    worker.start()
    demo = build_ui()
    demo.queue(default_concurrency_limit=8)

    user, password = os.environ.get("GRADIO_USER", "admin"), os.environ.get("GRADIO_PASSWORD", "")
    demo.launch(
        server_name="0.0.0.0",
        server_port=int(os.environ.get("AUTOTRADUCTOR_PORT", "7860")),
        share=os.environ.get("AUTOTRADUCTOR_SHARE", "1") == "1",
        auth=(user, password) if password else None,
        allowed_paths=[str(DATA_DIR.resolve())],
        show_api=False,
        show_error=True,
    )


if __name__ == "__main__":
    main()


## 8. Arranque (mantener en ejecución)
Al terminar verás la URL pública.

In [ ]:
import re, sys, os, time, subprocess, pathlib

# --- Verificación inteligente y guiada de pasos previos ---
_missing = []
_app_dir = globals().get("APP_DIR")
if not _app_dir or not pathlib.Path(str(_app_dir)).is_dir():
    _missing.append("• Celda 1 (Configuración): define los directorios y detecta la GPU.")
if _app_dir and not (pathlib.Path(str(_app_dir)) / "gradio_app.py").is_file():
    _missing.append("• Celda 7 (Código de la app): genera los archivos del backend (gradio_app.py, pipeline.py, etc.).")

if _missing:
    print("=" * 74)
    print("⚠️ FALTAN PASOS PREVIOS POR EJECUTAR EN ESTE NOTEBOOK")
    print("=" * 74)
    print("Para que AutoTraductor pueda arrancar, debes ejecutar primero:")
    for step in _missing:
        print(f"  {step}")
    print("\n💡 RECOMENDACIÓN:")
    print("   En el menú superior de Kaggle, haz clic en: 'Run' ➔ 'Run All'")
    print("   (o ejecuta las celdas en orden de arriba a abajo).")
    print("=" * 74)
    raise SystemExit(1)

# Comprobación de Qwen3-TTS con guía informativa (evita NameError críptico)
if "QWEN_TTS_READY" not in globals():
    print("=" * 74)
    print("ℹ️ GUÍA: La Celda 4 (Qwen3-TTS) no se ejecutó en este kernel.")
    print("   -> AutoTraductor continuará con XTTS v2 como motor de clonación predeterminado.")
    print("   -> Si deseas habilitar Qwen3-TTS además de XTTS v2, ejecuta la Celda 4 y vuelve a iniciar esta celda.")
    print("=" * 74)
    _qwen_ready = False
    _qwen_pkg = pathlib.Path("/kaggle/tmp/autotraductor-qwen-package")
    _qwen_err = "Celda 4 no ejecutada en este kernel"
else:
    _qwen_ready = QWEN_TTS_READY
    _qwen_pkg = QWEN_TTS_PACKAGE
    _qwen_err = QWEN_TTS_ERROR

_data_dir = globals().get("DATA_DIR", pathlib.Path("/kaggle/working/autotraductor"))
_storage_limit = globals().get("STORAGE_LIMIT_GB", 15)
_cookies_file = globals().get("COOKIES_FILE", pathlib.Path("/kaggle/tmp/yt_cookies.txt"))
_hf_home = globals().get("HF_HOME", pathlib.Path("/kaggle/tmp/hf-cache"))
_secrets = globals().get("SECRETS", {})

def _kill(pattern):
    subprocess.run(["pkill", "-f", pattern], capture_output=True)

_kill("gradio_app.py")
time.sleep(1)

env = dict(os.environ)
env.update({
    "AUTOTRADUCTOR_DATA_DIR": str(_data_dir),
    "AUTOTRADUCTOR_STORAGE_LIMIT_GB": str(_storage_limit),
    "AUTOTRADUCTOR_YT_COOKIES_FILE": str(_cookies_file),
    "AUTOTRADUCTOR_PRELOAD": "0",
    "AUTOTRADUCTOR_QWEN_TTS_READY": "1" if _qwen_ready else "0",
    "AUTOTRADUCTOR_QWEN_TTS_PACKAGE": str(_qwen_pkg),
    "AUTOTRADUCTOR_QWEN_TTS_ERROR": str(_qwen_err)[:1200],
    "AUTOTRADUCTOR_SHARE": "1",
    "HF_HOME": str(_hf_home),
    "COQUI_TOS_AGREED": "1",
    "PYTHONUNBUFFERED": "1",
    "GRADIO_ANALYTICS_ENABLED": "False",
})
if _secrets.get("GRADIO_PASSWORD"):
    env["GRADIO_PASSWORD"] = _secrets["GRADIO_PASSWORD"]
if _secrets.get("KAGGLE_USERNAME") and _secrets.get("KAGGLE_KEY"):
    env["KAGGLE_USERNAME"], env["KAGGLE_KEY"] = _secrets["KAGGLE_USERNAME"], _secrets["KAGGLE_KEY"]

APP_LOG = pathlib.Path("/kaggle/tmp/gradio_app.log")
URL_RE = re.compile(r"https://[a-z0-9\-]+\.gradio\.live")

def start_app():
    return subprocess.Popen(
        [sys.executable, "gradio_app.py"], cwd=_app_dir, env=env,
        stdout=open(APP_LOG, "w"), stderr=subprocess.STDOUT,
    )

def wait_for_url(proc, timeout=180):
    for _ in range(timeout):
        if proc.poll() is not None:
            return None
        match = URL_RE.search(APP_LOG.read_text())
        if match:
            return match.group(0)
        time.sleep(1)
    return None

def show_url(url):
    auth = "usuario: admin · clave: tu secret GRADIO_PASSWORD" if _secrets.get("GRADIO_PASSWORD") else "SIN CLAVE: cualquiera con este enlace puede usarla (define el secret GRADIO_PASSWORD)"
    line = "=" * 74
    print(f"\n{line}\n  AutoTraductor listo ->  {url}\n  Acceso: {auth}\n{line}\n")

app_proc = start_app()
public_url = wait_for_url(app_proc)
if public_url is None:
    print(APP_LOG.read_text()[-3000:])
    raise RuntimeError("Gradio no generó la URL pública. Revisa el log de arriba (¿Internet activado? ¿falló una dependencia?).")
show_url(public_url)
print("Mantén esta celda en ejecución (no la detengas). La web se actualiza sola.")

last_beat = time.time()
restarts = 0
while True:
    time.sleep(30)
    if app_proc.poll() is not None:
        if restarts >= 5:
            print(APP_LOG.read_text()[-3000:])
            raise RuntimeError("La app se detuvo repetidamente. Revisa el log de arriba.")
        restarts += 1
        print(f"[{time.strftime('%H:%M:%S')}] La app se detuvo; reiniciando {restarts}/5 (los trabajos se reanudan)…")
        print(APP_LOG.read_text()[-1500:])
        app_proc = start_app()
        new_url = wait_for_url(app_proc)
        if new_url:
            show_url(new_url)
    if time.time() - last_beat > 600:
        last_beat = time.time()
        print(f"[{time.strftime('%H:%M:%S')}] activo · PID {app_proc.pid} · {public_url if app_proc.poll() is None else ''}")
